# SAM 2.1 Hiera-Small — DIMER end-to-end promptable segmentation fine-tuning

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/sam2-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/sam2-segmentation-pipeline/blob/main/tutorials/sam2_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fsam2.1--hiera--small-ffcc4d?style=flat)](https://huggingface.co/facebook/sam2.1-hiera-small)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** pinned SAM 2.1 inference plus bounded mask-hypernetwork gradient adaptation, held-out mask-IoU evaluation, safe adapter export, and fresh reload

**This notebook is standalone.** Section 2 carries the repository's package (2 modules under `src/sam2_segmentation_pipeline/`, at revision `f60fa852d6b0`), the stage runner, the hash-locked requirements (47 packages) and the pinned checkpoint manifest, and verifies every carried file against its SHA-256 before use, so the notebook keeps working after export even if the repository changes or disappears. Nothing is installed into the notebook kernel: a pinned `uv` (checked by size and SHA-256) builds an isolated Python environment from the lock with `--require-hashes`, and every stage runs there in its own process, so the hosted runtime's own packages are never replaced and no restart is needed. Its only external dependencies are PyPI, the managed CPython build that `uv` downloads, and the Hugging Face Hub at the immutable revision `ee5bba1d82bb8749febdf90f45e84b687142ba03` (~184 MB in all, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/3.0); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** on a fresh Linux x86_64 runtime with a CUDA GPU (Colab T4 or Kaggle GPU) builds an isolated Python environment from the carried hash-locked requirements without touching the notebook kernel's own packages, verifies the exact base snapshot, generates and validates 24 records, measures the pretrained held-out baseline under two prompt conditions, freezes the base and performs the bounded gradient update from the pinned base, exports the adapter, attaches it to a fresh base to score the held-out split per record, predicts an unseen scene with a reload check against the adapting process, and writes provenance — without a clone, credential, upload, configuration edit or runtime restart. On a CPU-only runtime Section 1 stops at once with the runtime-change instruction.

**Bring Your Own Data:** Set `USE_BYOD = True` and either set `BYOD_ZIP_PATH` to a ZIP in this runtime (any runtime; an attached Kaggle input is found automatically when it is the only ZIP under `/kaggle/input/`) or leave it empty in Colab to open the upload dialog. The ZIP holds `images/<stem>.png|jpg` and `masks/<stem>.png` (white = foreground), at the top level or inside one wrapping folder; at least 8 pairs, at most 64. The held-out split is a seeded shuffle; set `GROUP_SEPARATOR` (for example `_`) so frames or crops of one scene (`scene1_a`, `scene1_b`) stay on one side. Your records pass through the same validation, baseline, adaptation, evaluation, export and reload path, and the unseen inference and reload check run on one of your held-out records. Every refusal names the archive and the rule.

This notebook performs a real local gradient update. It freezes the Hiera encoder, prompt encoder, and the rest of the mask decoder, then trains only the first mask-token output hypernetwork (139,808 parameters) with BCE plus soft-Dice loss. The default dataset is 24 deterministic generated shape scenes split 18/6 before model execution. A SafeTensors adapter and closed SHA-256 manifest are exported and attached to a fresh pinned base model. These are tutorial sample-sanity results, not a segmentation benchmark.

**Read this before Section 8: the default task leaves almost nothing to adapt.** Each generated scene is one bright shape on a flat dark background, and the recorded condition gives the model the shape's exact box and an interior point. The pretrained model already scored held-out mask IoU 0.998254 in the recorded run, and the adapted model 0.998457 — a change of +0.0002 on six images, which is a mechanics check, not an improvement. The notebook also scores a second held-out condition, **point-only prompts** (no box), generated from the same records, and prints every held-out record before and after. One interior point is enough on a single high-contrast shape, so point-only is at ceiling too (about 0.998 mean IoU before and after adaptation in a local check of the pinned checkpoint). Neither condition can show an adaptation effect: the lesson here is the workflow — validate, measure, adapt, export, reload, and read the result honestly — not a gain.

**Learning objectives:** by the end of this notebook you will be able to —

1. **Validate** paired image / mask / prompt records and **explain** why the held-out split is fixed before any model runs (Sections 4, 5).
2. **Measure** a pretrained held-out baseline under two prompt conditions and **say** how much headroom it leaves (Section 6).
3. **Describe** what the bounded fine-tune changes — one mask hypernetwork — and why the weights moving is not evidence of improvement (Section 7).
4. **Judge** an adaptation result from per-record held-out IoU, and **distinguish** a mechanics check from evidence of improvement (Section 8).
5. **Verify** that the exported adapter on a fresh base reproduces the adapting process (Section 9).
6. **Predict**, run and **explain** the effect of a larger learning rate in an optional activity (Section 11).

**This notebook does not demonstrate:** full-model training, video tracking, automatic segment-everything, semantic class learning, production-scale training, or benchmark claims. The adapter changes one image-mode mask hypernetwork only.

## How to use this notebook

**Who this notebook is for.** Learners who can run cells in a hosted notebook and read short Python, who know what a binary mask and IoU are, and who want to see a promptable segmentation model adapted end to end: a fixed held-out split, a pretrained baseline, a bounded fine-tune, an honest per-record comparison, and an adapter that reloads with identical masks. No experience with SAM is assumed; the glossary below explains every term.

**Running it.** Choose a GPU runtime (*Runtime → Change runtime type → T4 GPU* in Colab), then *Runtime → Run all*. The default path needs no edit, no upload, no account, no token and no runtime restart. Section 2 builds an isolated environment, which takes the longest.

**Where the code runs.** The notebook kernel installs nothing and imports no model library. Each learner cell calls `run_stage('…')`, which runs one stage of the carried stage runner in its own process with the isolated environment's Python, streams what it prints, and stops the notebook with the stage's own error message if it fails. Stages hand results to each other only through files; the adapted model reaches Sections 8–10 only through the exported adapter.

**Two kinds of cell.** *Learner cells* (Sections 4–11) are the workflow. *Infrastructure cells* (Sections 1–3) are collapsed and titled **Infrastructure**; you may run them without studying their implementation.

**Form controls.** `USE_BYOD`, `BYOD_ZIP_PATH`, `GROUP_SEPARATOR` and `SEED` (Section 4); `EPOCHS` and `LEARNING_RATE` (Section 7); `RUN_ACTIVITY` and `ACTIVITY_LEARNING_RATE` (Section 11). Leave them at their defaults for the first run.

**Section tags.** **[Concept]** — what the model does and why. **[Evaluation practice]** — how the evidence is produced and how to read it. **[Engineering]** — reproducibility, provenance and packaging.

**Predict, then check.** Before Sections 6, 7 and 8 a **Predict before running** prompt asks you to commit to an expectation; **What to notice** follows each stage; a collapsed **Check your reasoning** answer follows each checkpoint. The sample answers quote the recorded Kaggle T4 run for the point+box condition.

## The task: Input → Model → Output

| Stage | Input | Model / system | Output |
|---|---|---|---|
| **Validate and split** | 24 images, each with a mask, an interior point and a box | `validate_segmentation_dataset`; fixed 18 / 6 split | a dataset manifest with content fingerprints |
| **Baseline** | the 6 held-out records, two prompt conditions | pretrained SAM 2.1 Hiera-Small (image mode) | per-record mask IoU |
| **Adapt** | the 18 training records | AdamW on one mask hypernetwork (BCE + soft-Dice), base frozen | an adapter (`adapter.safetensors` + manifest) |
| **Evaluate** | the held-out records again | the adapter on a fresh base, in a fresh process | per-record change, records improved, an interpretation |
| **Reuse** | an unseen scene | the adapter on a fresh base | a mask; reload parity with the adapting process |

## Roadmap

| Section | Tag | What happens | What you read |
|---|---|---|---|
| 1–3 | [Engineering] | GPU check, carried code, isolated environment, verified snapshot | versions, digests |
| 4. Data | [Evaluation practice] | generated or BYOD records, the split | counts, split rule |
| 5. Validation | [Evaluation practice] | dataset contract and leakage check | the dataset manifest |
| 6. Baseline | [Evaluation practice] | pretrained IoU, two prompt conditions | the headroom |
| 7. Fine-tune | [Concept] | bounded update from the pinned base; adapter exported | history, weight delta |
| 8. Evaluation | [Evaluation practice] | per-record before/after, both conditions | records improved, interpretation |
| 9. Unseen scene and reload | [Engineering] | inference and reload parity across processes | IoU, `verification` |
| 10. Provenance | [Engineering] | the result record | `result.json` |
| 11. Optional activity | [Concept] | a larger learning rate, printed beside the default | a comparison table |
| Troubleshooting | [Engineering] | every failure and what to do | when something fails |
| Interpretation and conclusion | [Evaluation practice] | what was and was not shown | your conclusion |

<details>
<summary><strong>Glossary</strong> — open when a term is unfamiliar</summary>

| Term | Meaning in this notebook |
|---|---|
| **Promptable segmentation** | The model returns a mask for the object a prompt points at, rather than labelling classes. |
| **Prompt (point / box)** | A clicked point inside the object and/or a box around it; this notebook compares point+box with point-only. |
| **Mask decoder / mask token** | The small network that turns image features and prompts into masks; one learned token per output mask. |
| **Hypernetwork** | A small MLP that produces the weights used to read out one mask token; the only part trained here. |
| **Mask IoU** | Intersection over union of predicted and true masks: 1.0 is a perfect match. |
| **Soft-Dice loss** | A differentiable overlap loss used with binary cross-entropy (BCE) during training. |
| **Headroom** | How far the pretrained score is from perfect; with no headroom, adaptation cannot show an effect. |
| **Adapter vs base** | The adapter holds only the trained hypernetwork; it is attached to the pinned, unchanged base model. |
| **Reload parity** | A fresh base plus the adapter gives the same masks and scores as the process that trained it. |
| **Hash-locked environment / stage** | The isolated Python environment every stage runs in; one workflow step run as its own process. |

</details>

## Prerequisites

- **Runtime:** a fresh **Linux x86_64** runtime **with a CUDA GPU** — Google Colab with a T4 GPU, or a Kaggle GPU kernel. The fine-tune needs CUDA, so Section 1 stops at once on a CPU-only runtime. The kernel's own Python version does not matter: the notebook installs nothing into it, and runs every stage with CPython 3.12.12 in an isolated environment built from 47 hash-locked packages (`torch` 2.14.0 with its CUDA libraries, `transformers` 4.57.6, `safetensors` 0.8.0, `numpy` 2.5.3, `pillow` 11.3.0). About 0.2 GB of disk is needed for the snapshot and about 9 GB for the isolated environment.
- **Knowledge:** Python, binary masks and IoU, and why training and held-out data must be kept apart. The glossary below explains the SAM-specific terms.
- **Data:** the default path generates 24 shape scenes. Optional BYOD accepts one ZIP with `images/` and `masks/` paired by stem (masks as PNG, white = foreground), at least 8 and at most 64 pairs, each image 16..4096 px per side and matching its mask, 32 megapixels in all. Do not upload confidential or restricted data unless you are authorized to use it in the hosted runtime.
- **External access:** `huggingface.co` (no account or token), to fetch the pinned `facebook/sam2.1-hiera-small` checkpoint (~184 MB) at revision `ee5bba1d82bb…`, every file accepted only at the pinned size and SHA-256; PyPI (`pypi.org`, `files.pythonhosted.org`), for the pinned `uv` wheel and the 47 hash-locked packages; and the managed CPython build (python-build-standalone) that `uv` downloads for the isolated environment. No repository clone and no credentials are required; nothing is installed from this repository.

## 1. Check the runtime · [Engineering]

> **Infrastructure.** The code cells in Sections 1–3 are collapsed and titled **Infrastructure**. You may run them without studying their implementation; they exist for reproducibility and provenance, not as prerequisite machine-learning knowledge. The learning activities start in Section 4.

**Input:** a fresh hosted runtime. **System:** checks that it is Linux x86_64 with enough free disk, looks for a CUDA GPU, and creates a run directory under `dimer_runs/sam2_segmentation/` for the carried code, the stage logs and the hand-off state between stages. Learner-facing files are written to `outputs/`. **Output:** one dictionary naming the accelerator, the kernel's Python version and the directories. Running this cell again in the same session keeps the run directory, so the cells after it keep working; tick `NEW_RUN_DIRECTORY` for a fresh one and run Sections 2 and 3 again. The verified checkpoint is kept in `weights/` and reused.

In [ ]:
# @title Infrastructure: check the runtime, accelerator and disk; create (or keep) the run directory
NEW_RUN_DIRECTORY = False  # @param {type:"boolean"}
import base64
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tempfile
import time
import uuid
from pathlib import Path

SESSION_START = time.perf_counter()
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle, or a Linux x86_64 Jupyter kernel): its locked environment is built for manylinux x86_64 wheels. A GPU is recommended, not required.')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    accelerator = gpu.stdout.strip() if gpu.returncode == 0 else ''
except FileNotFoundError:
    accelerator = ''
if not accelerator:
    raise RuntimeError('No CUDA GPU detected, and this notebook\'s fine-tuning needs one: choose Runtime > Change runtime type > T4 GPU (Colab) or turn on a GPU accelerator (Kaggle), then Run all. Nothing has been downloaded yet.')
STEM = 'sam2_segmentation'
# Re-running this cell keeps the run directory of this session, so the cells after it keep working; tick
# NEW_RUN_DIRECTORY (then run the cells below again) for a fresh one.
if NEW_RUN_DIRECTORY or not isinstance(globals().get('ROOT'), Path) or not globals()['ROOT'].is_dir():
    ROOT = Path.cwd() / 'dimer_runs' / STEM / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUTS = Path.cwd() / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
WEIGHTS = Path.cwd() / 'weights'
WEIGHTS.mkdir(exist_ok=True)
ENV_ROOT = Path(tempfile.gettempdir()) / 'dimer_uv'
staged_gib = sum(p.stat().st_size for p in WEIGHTS.rglob('*') if p.is_file()) / 1024**3
need = {'weights': max(0.0, 0.2 - staged_gib), 'environment': 9.0}
free = {'weights': shutil.disk_usage(WEIGHTS).free / 1024**3, 'environment': shutil.disk_usage(tempfile.gettempdir()).free / 1024**3}
if os.stat(WEIGHTS).st_dev == os.stat(tempfile.gettempdir()).st_dev:
    short = free['weights'] < need['weights'] + need['environment']
else:
    short = free['weights'] < need['weights'] or free['environment'] < need['environment']
if short:
    raise RuntimeError(f'Not enough free disk: need about {need} GiB, free {free} GiB. Start a fresh runtime (see Troubleshooting).')
print({'accelerator': accelerator, 'kernel_python': platform.python_version(), 'run_directory': str(ROOT), 'outputs': str(OUTPUTS), 'weights': str(WEIGHTS), 'environment_root': str(ENV_ROOT), 'free_gib': {k: round(v, 1) for k, v in free.items()}})

{'accelerator': 'Tesla T4, 15360 MiB', 'kernel_python': '3.13.15', 'run_directory': '/content/dimer_runs/sam2_segmentation/8da0291f3dad', 'outputs': '/content/outputs', 'weights': '/content/weights', 'environment_root': '/tmp/dimer_uv', 'free_gib': {'weights': 193.3, 'environment': 193.3}}


## 2. Carry the code and build the isolated environment · [Engineering]

> **Infrastructure.** The next two code cells are collapsed. The first **is** the repository's code, carried so that this notebook works on its own; the second builds the environment every stage runs in.

The first cell holds, as text, the files the workflow needs: the package's 2 modules under `src/sam2_segmentation_pipeline/` (at revision `f60fa852d6b0`), the stage runner `tutorial_stages.py`, the hash-locked `requirements.txt` (47 packages), the pinned checkpoint manifest and the licence. It writes each file into the run directory and checks its SHA-256 against `CARRIED_HASHES`, stopping on any mismatch. The text is the repository's files byte for byte; the repository's parity tests fail whenever the two diverge, so what runs here is what the repository tests. Nothing in this cell runs a model.

**Expected result:** `carried_files`, `verified: True`, and the repository revision the notebook was generated from.

In [ ]:
# @title Infrastructure: write and verify the carried package, stage runner, lock, manifest and samples
CARRIED_FILES = {'src/sam2_segmentation_pipeline/__init__.py': 'from .pipeline import (\n    DEFAULT_WEIGHTS_DIR,\n    INPUT_SCHEMA,\n    MAX_ADAPTATION_PIXELS,\n    MAX_IMAGE_SIDE,\n    MAX_PROMPTS,\n    MIN_IMAGE_SIDE,\n    MODEL_ID,\n    MODEL_KEY,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    NUM_MULTIMASK_OUTPUTS,\n    SAM2SegmentationPipeline,\n    evaluation_report,\n    mask_iou,\n    stage_missing_files,\n    validate_inputs,\n    validate_segmentation_dataset,\n    verify_snapshot,\n)\n\n__all__ = [\n    "DEFAULT_WEIGHTS_DIR",\n    "INPUT_SCHEMA",\n    "MAX_ADAPTATION_PIXELS",\n    "MAX_IMAGE_SIDE",\n    "MAX_PROMPTS",\n    "MIN_IMAGE_SIDE",\n    "MODEL_ID",\n    "MODEL_KEY",\n    "MODEL_LICENSE",\n    "MODEL_REVISION",\n    "NUM_MULTIMASK_OUTPUTS",\n    "SAM2SegmentationPipeline",\n    "evaluation_report",\n    "mask_iou",\n    "stage_missing_files",\n    "validate_inputs",\n    "validate_segmentation_dataset",\n    "verify_snapshot",\n]\n', 'src/sam2_segmentation_pipeline/pipeline.py': 'from __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\nimport math\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom datetime import UTC, datetime\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nfrom PIL import Image\n\nMODEL_ID = "facebook/sam2.1-hiera-small"\nMODEL_REVISION = "ee5bba1d82bb8749febdf90f45e84b687142ba03"\nMODEL_LICENSE = "apache-2.0"\nMODEL_KEY = "sam2.1-hiera-small"\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\nMANIFEST_NAME = "dimer-base-manifest.json"\nARTIFACT_FORMAT = "org.valcorza.sam2.mask-decoder-adapter"\nARTIFACT_FORMAT_VERSION = "1.0"\nARTIFACT_MANIFEST_NAME = "manifest.json"\nARTIFACT_WEIGHTS_NAME = "adapter.safetensors"\nTRAINABLE_PREFIXES = ("mask_decoder.output_hypernetworks_mlps.0.",)\nADAPTATION_METHOD = "frozen-backbone-mask-hypernetwork-gradient-adaptation"\nMAX_ADAPTATION_RECORDS = 128\nMAX_ADAPTATION_PIXELS = 32 * 1024 * 1024\n\n_ADAPTATION_RUN_FIELDS = (\n    "epochs",\n    "learning_rate",\n    "batch_size",\n    "seed",\n    "loss",\n    "train_manifest",\n    "validation_manifest",\n    "baseline_validation_mask_iou",\n    "history",\n    "weight_delta_l2",\n)\n_DATASET_MANIFEST_FIELDS = {\n    "records",\n    "unique_ids",\n    "positive_pixels",\n    "dataset_sha256",\n    "verdict",\n}\n_ADAPTATION_BASE_FIELDS = {\n    "method",\n    "trainable_prefixes",\n    "trainable_parameters",\n    "frozen_parameters",\n}\n_ADAPTATION_METADATA_FIELDS = {\n    *_ADAPTATION_BASE_FIELDS,\n    *_ADAPTATION_RUN_FIELDS,\n    "training_runs",\n    "artifact_lineage",\n}\n_ARTIFACT_LINEAGE_FIELDS = {"manifest_sha256", "weights_sha256", "producer"}\n\n# Input ceilings. The processor resizes every image to 1024x1024 (preprocessor_config.json), so model cost is\n# fixed; the caller\'s resolution only sets the size of the up-sampled output masks. Prompts are one object per\n# call: up to MAX_PROMPTS point clicks (label 1 = foreground, 0 = background) and/or one xyxy box.\nMAX_IMAGE_SIDE = 4096\nMIN_IMAGE_SIDE = 16\nMAX_PROMPTS = 16\nNUM_MULTIMASK_OUTPUTS = 3  # config.json mask_decoder_config.num_multimask_outputs\nMASK_THRESHOLD = 0.0  # logits above this become True in the binarised masks (processor default)\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for chunk in iter(lambda: fh.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _is_positive_finite_number(value: Any) -> bool:\n    return (\n        isinstance(value, int | float)\n        and not isinstance(value, bool)\n        and math.isfinite(float(value))\n        and value > 0\n    )\n\n\ndef _validate_dataset_manifest(manifest: Any, *, label: str) -> None:\n    if not isinstance(manifest, Mapping):\n        raise ValueError(f"{label} must be a mapping")\n    if set(manifest) != _DATASET_MANIFEST_FIELDS:\n        raise ValueError(f"{label} does not match the dataset manifest schema")\n    records = manifest["records"]\n    unique_ids = manifest["unique_ids"]\n    positive_pixels = manifest["positive_pixels"]\n    if (\n        not isinstance(records, int)\n        or isinstance(records, bool)\n        or not 2 <= records <= MAX_ADAPTATION_RECORDS\n    ):\n        raise ValueError(f"{label} has an invalid record count")\n    if not isinstance(unique_ids, int) or isinstance(unique_ids, bool) or unique_ids != records:\n        raise ValueError(f"{label} has an invalid unique-id count")\n    if (\n        not isinstance(positive_pixels, int)\n        or isinstance(positive_pixels, bool)\n        or positive_pixels <= 0\n    ):\n        raise ValueError(f"{label} has an invalid positive-pixel count")\n    digest = manifest["dataset_sha256"]\n    if (\n        not isinstance(digest, str)\n        or len(digest) != 64\n        or any(character not in "0123456789abcdef" for character in digest)\n    ):\n        raise ValueError(f"{label} has an invalid dataset SHA-256")\n    if manifest["verdict"] != "accepted":\n        raise ValueError(f"{label} has an invalid verdict")\n\n\ndef _validate_adaptation_run(run: Any, *, label: str) -> None:\n    if not isinstance(run, Mapping):\n        raise ValueError(f"{label} must be a mapping")\n    if set(run) != set(_ADAPTATION_RUN_FIELDS):\n        raise ValueError(f"{label} does not match the adaptation run schema")\n    missing = [key for key in _ADAPTATION_RUN_FIELDS if key not in run]\n    if missing:\n        raise ValueError(f"{label} is missing required fields: {missing}")\n    epochs = run["epochs"]\n    batch_size = run["batch_size"]\n    seed = run["seed"]\n    if not isinstance(epochs, int) or isinstance(epochs, bool) or epochs <= 0:\n        raise ValueError(f"{label} has invalid epochs")\n    if not _is_positive_finite_number(run["learning_rate"]):\n        raise ValueError(f"{label} has invalid learning rate")\n    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size <= 0:\n        raise ValueError(f"{label} has invalid batch size")\n    if not isinstance(seed, int) or isinstance(seed, bool):\n        raise ValueError(f"{label} has invalid seed")\n    if run["loss"] != "binary-cross-entropy-plus-soft-dice":\n        raise ValueError(f"{label} has an unsupported loss")\n    _validate_dataset_manifest(run["train_manifest"], label=f"{label} train manifest")\n    validation_manifest = run["validation_manifest"]\n    if validation_manifest is not None:\n        _validate_dataset_manifest(\n            validation_manifest, label=f"{label} validation manifest"\n        )\n    baseline = run["baseline_validation_mask_iou"]\n    if validation_manifest is None and baseline is not None:\n        raise ValueError(f"{label} has invalid validation baseline")\n    if baseline is not None and (\n        not isinstance(baseline, int | float)\n        or isinstance(baseline, bool)\n        or not math.isfinite(float(baseline))\n        or not 0 <= float(baseline) <= 1\n    ):\n        raise ValueError(f"{label} has invalid validation baseline")\n    history = run["history"]\n    if not isinstance(history, list) or not history:\n        raise ValueError(f"{label} requires non-empty training history")\n    if len(history) != epochs:\n        raise ValueError(f"{label} history length does not match epochs")\n    expected_history_fields = {"epoch", "train_loss", "optimizer_steps"}\n    if validation_manifest is not None:\n        expected_history_fields.add("val_mask_iou")\n    expected_steps = math.ceil(run["train_manifest"]["records"] / batch_size)\n    for expected_epoch, item in enumerate(history, start=1):\n        if not isinstance(item, Mapping) or set(item) != expected_history_fields:\n            raise ValueError(f"{label} history entry schema is invalid")\n        if item["epoch"] != expected_epoch or isinstance(item["epoch"], bool):\n            raise ValueError(f"{label} history has an invalid epoch sequence")\n        train_loss = item["train_loss"]\n        if (\n            not isinstance(train_loss, int | float)\n            or isinstance(train_loss, bool)\n            or not math.isfinite(float(train_loss))\n            or train_loss < 0\n        ):\n            raise ValueError(f"{label} history has an invalid train loss")\n        optimizer_steps = item["optimizer_steps"]\n        if (\n            not isinstance(optimizer_steps, int)\n            or isinstance(optimizer_steps, bool)\n            or optimizer_steps != expected_steps\n        ):\n            raise ValueError(f"{label} history has an invalid optimizer step count")\n        if validation_manifest is not None:\n            val_mask_iou = item["val_mask_iou"]\n            if (\n                not isinstance(val_mask_iou, int | float)\n                or isinstance(val_mask_iou, bool)\n                or not math.isfinite(float(val_mask_iou))\n                or not 0 <= float(val_mask_iou) <= 1\n            ):\n                raise ValueError(f"{label} history has an invalid validation mask IoU")\n    if not _is_positive_finite_number(run["weight_delta_l2"]):\n        raise ValueError(f"{label} requires a positive finite weight delta")\n\n\ndef _validate_completed_adaptation_metadata(adaptation: Any) -> None:\n    """Reject metadata that cannot identify a completed adapter before weights are applied."""\n    if not isinstance(adaptation, Mapping):\n        raise ValueError("artifact adaptation metadata must be a mapping")\n    unknown_fields = set(adaptation) - _ADAPTATION_METADATA_FIELDS\n    if unknown_fields:\n        raise ValueError(\n            f"artifact adaptation metadata contains unknown fields: {sorted(unknown_fields)}"\n        )\n    if adaptation.get("method") != ADAPTATION_METHOD:\n        raise ValueError("artifact adaptation metadata has an unsupported or missing method")\n    if adaptation.get("trainable_prefixes") != list(TRAINABLE_PREFIXES):\n        raise ValueError("artifact adaptation metadata has incomplete trainable prefixes")\n    trainable = adaptation.get("trainable_parameters")\n    frozen = adaptation.get("frozen_parameters")\n    if not isinstance(trainable, int) or isinstance(trainable, bool) or trainable <= 0:\n        raise ValueError("artifact adaptation metadata has invalid trainable parameter count")\n    if not isinstance(frozen, int) or isinstance(frozen, bool) or frozen < 0:\n        raise ValueError("artifact adaptation metadata has invalid frozen parameter count")\n    _validate_adaptation_run(\n        _adaptation_run_snapshot(adaptation),\n        label="artifact adaptation metadata current run",\n    )\n    if "training_runs" in adaptation:\n        training_runs = adaptation["training_runs"]\n        if not isinstance(training_runs, list) or not training_runs:\n            raise ValueError("artifact adaptation metadata training_runs must be a non-empty list")\n        for index, run in enumerate(training_runs):\n            _validate_adaptation_run(run, label=f"artifact adaptation metadata training_runs[{index}]")\n        if training_runs[-1] != _adaptation_run_snapshot(adaptation):\n            raise ValueError("artifact adaptation metadata latest training run does not match current run")\n    if "artifact_lineage" in adaptation:\n        artifact_lineage = adaptation["artifact_lineage"]\n        if not isinstance(artifact_lineage, list):\n            raise ValueError("artifact adaptation metadata artifact_lineage must be a list of mappings")\n        for index, item in enumerate(artifact_lineage):\n            if not isinstance(item, Mapping) or set(item) != _ARTIFACT_LINEAGE_FIELDS:\n                raise ValueError(\n                    f"artifact adaptation metadata artifact_lineage[{index}] has invalid fields"\n                )\n            for digest_field in ("manifest_sha256", "weights_sha256"):\n                digest = item[digest_field]\n                if (\n                    not isinstance(digest, str)\n                    or len(digest) != 64\n                    or any(character not in "0123456789abcdef" for character in digest)\n                ):\n                    raise ValueError(\n                        f"artifact adaptation metadata artifact_lineage[{index}] has an invalid digest"\n                    )\n            producer = item["producer"]\n            if (\n                not isinstance(producer, Mapping)\n                or set(producer) != {"pipelineId", "revision"}\n                or producer["pipelineId"] != "sam2-segmentation-pipeline"\n                or not isinstance(producer["revision"], str)\n                or len(producer["revision"]) != 40\n                or any(\n                    character not in "0123456789abcdef" for character in producer["revision"]\n                )\n            ):\n                raise ValueError(\n                    f"artifact adaptation metadata artifact_lineage[{index}] has an invalid producer"\n                )\n\n\ndef _adaptation_run_snapshot(adaptation: Mapping[str, Any]) -> dict[str, Any]:\n    """Copy the per-run fields retained in cumulative adapter provenance."""\n    return {\n        key: json.loads(json.dumps(adaptation[key]))\n        for key in _ADAPTATION_RUN_FIELDS\n        if key in adaptation\n    }\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as fh:\n        manifest = json.load(fh)\n    if manifest.get("modelId") != MODEL_ID:\n        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n    if manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n    for entry in manifest["files"]:\n        file_path = root / entry["path"]\n        if not file_path.is_file():\n            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n        size = file_path.stat().st_size\n        if size != entry["bytes"]:\n            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n        digest = _sha256(file_path)\n        if digest != entry["sha256"]:\n            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n    return {\n        "path": str(root),\n        "model_id": manifest["modelId"],\n        "revision": manifest["revision"],\n        "files": len(manifest["files"]),\n        "total_bytes": manifest.get("totalBytes"),\n    }\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as fh:\n        manifest = json.load(fh)\n    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(\n            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n        )\n    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f"snapshot at {root} is missing {missing}; "\n            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef mask_iou(a: np.ndarray, b: np.ndarray) -> float:\n    """Intersection-over-union of two boolean masks of identical shape; the primitive behind any mIoU."""\n    a = np.asarray(a)\n    b = np.asarray(b)\n    if a.shape != b.shape:\n        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")\n    if a.dtype != np.bool_ or b.dtype != np.bool_:\n        raise TypeError("mask_iou expects boolean arrays")\n    union = np.logical_or(a, b).sum()\n    return float(np.logical_and(a, b).sum() / union) if union else 0.0\n\n\ndef validate_image(image: Any) -> Image.Image:\n    if not isinstance(image, Image.Image):\n        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")\n    width, height = image.size\n    if min(width, height) < MIN_IMAGE_SIDE:\n        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")\n    if max(width, height) > MAX_IMAGE_SIDE:\n        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")\n    return image.convert("RGB")\n\n\ndef validate_prompts(\n    width: int,\n    height: int,\n    points: Sequence[Sequence[float]] | None,\n    point_labels: Sequence[int] | None,\n    box: Sequence[float] | None,\n) -> tuple[list[list[float]] | None, list[int] | None, list[float] | None]:\n    """Check one object\'s prompts: points inside the image with 0/1 labels, and/or one xyxy box inside it."""\n    if points is None and box is None:\n        raise ValueError("provide at least one of points or box")\n    clean_points = clean_labels = None\n    if points is not None:\n        if isinstance(points, str) or not isinstance(points, Sequence):\n            raise TypeError("points must be a sequence of [x, y] pairs")\n        if not 1 <= len(points) <= MAX_PROMPTS:\n            raise ValueError(f"point count {len(points)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")\n        if point_labels is None or len(point_labels) != len(points):\n            raise ValueError("point_labels must be given with one 0/1 entry per point")\n        clean_points, clean_labels = [], []\n        for (x, y), label in zip(points, point_labels, strict=True):\n            if not (0 <= x < width and 0 <= y < height):\n                raise ValueError(f"point ({x}, {y}) outside image {width}x{height}")\n            if label not in (0, 1) or isinstance(label, bool):\n                raise ValueError(f"point label must be 0 or 1, got {label!r}")\n            clean_points.append([float(x), float(y)])\n            clean_labels.append(int(label))\n    elif point_labels is not None:\n        raise ValueError("point_labels given without points")\n    clean_box = None\n    if box is not None:\n        if len(box) != 4:\n            raise ValueError("box must be [x0, y0, x1, y1]")\n        x0, y0, x1, y1 = (float(v) for v in box)\n        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):\n            raise ValueError(f"box {box!r} is not a non-empty xyxy box inside image {width}x{height}")\n        clean_box = [x0, y0, x1, y1]\n    return clean_points, clean_labels, clean_box\n\n\nINPUT_SCHEMA: dict[str, Any] = {\n    "input": (\n        "one PIL.Image.Image (any mode, converted to RGB) plus one object\'s prompts: point clicks "\n        "and/or one xyxy box"\n    ),\n    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n    "points": [1, MAX_PROMPTS],\n    "point_labels": "one per point, 1 = foreground and 0 = background",\n    "box": "at most one [x0, y0, x1, y1] inside the image with x0 < x1 and y0 < y1",\n    "adapted_multimask": (\n        "base pipelines default to three candidates; adapted pipelines default to one mask and reject "\n        "multimask=True because the adapter trains only the single-mask output hypernetwork"\n    ),\n    "objects_per_call": 1,\n    "multimask_outputs": NUM_MULTIMASK_OUTPUTS,\n    "preprocessing": (\n        "image converted to RGB; the processor resizes it to 1024x1024; returned masks are up-sampled "\n        f"to the input resolution and binarised at logit MASK_THRESHOLD={MASK_THRESHOLD}"\n    ),\n}\n\n\ndef _check_inputs(\n    image: Any,\n    points: Any,\n    point_labels: Any,\n    box: Any,\n    multimask: Any,\n) -> tuple[Image.Image, list[list[float]] | None, list[int] | None, list[float] | None]:\n    """Raise TypeError/ValueError naming the first violated ceiling; return the cleaned request.\n\n    ``segment`` and ``validate_inputs`` both route through this function so their acceptance\n    criteria cannot diverge.\n    """\n    rgb = validate_image(image)\n    clean_points, clean_labels, clean_box = validate_prompts(\n        rgb.width, rgb.height, points, point_labels, box\n    )\n    if not isinstance(multimask, bool):\n        raise TypeError("multimask must be a bool")\n    return rgb, clean_points, clean_labels, clean_box\n\n\ndef validate_inputs(\n    image: Image.Image,\n    *,\n    points: Sequence[Sequence[float]] | None = None,\n    point_labels: Sequence[int] | None = None,\n    box: Sequence[float] | None = None,\n    multimask: bool = True,\n    names: Sequence[str] | None = None,\n) -> dict[str, Any]:\n    """Validation stage: return the input manifest (schema, observations, request, verdict).\n\n    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding\n    recorded catches the exception and stores ``str(exc)`` under ``findings``.\n    """\n    _rgb, clean_points, clean_labels, clean_box = _check_inputs(\n        image, points, point_labels, box, multimask\n    )\n    if names is not None and len(names) != 1:\n        raise ValueError("names must have exactly one entry (segment takes one image)")\n    return {\n        "schema": dict(INPUT_SCHEMA),\n        "inputs": [\n            {\n                "id": names[0] if names else "image-0",\n                "mode": image.mode,\n                "size": list(image.size),\n                "n_points": 0 if clean_points is None else len(clean_points),\n                "has_box": clean_box is not None,\n            }\n        ],\n        "points": clean_points,\n        "point_labels": clean_labels,\n        "box": clean_box,\n        "multimask": multimask,\n        "verdict": "accepted",\n        "findings": [],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\ndef evaluation_report(\n    result: Mapping[str, Any],\n    reference_mask: Any = None,\n    *,\n    sample_kind: str = "synthetic",\n) -> dict[str, Any]:\n    """Evaluation stage: a machine-readable report even when nothing is measurable.\n\n    With a boolean ``reference_mask`` of the same shape as the returned masks the report carries one\n    ``mask_iou`` entry per candidate as sample-sanity geometry evidence; without one the verdict is\n    ``not-measurable`` and the report says what labelled data would make the task measurable.\n    """\n    masks = np.asarray(result["masks"])\n    scores = [float(v) for v in result["iou_scores"]]\n    best = int(np.argmax(scores)) if scores else None\n    base = {\n        "task": "promptable single-object image segmentation (point and/or box prompts)",\n        "decision_rule": (\n            "keep the candidate with the highest model-predicted IoU; the pipeline ships no "\n            "acceptance threshold and does not choose for the caller"\n        ),\n        "score_semantics": (\n            "iou_scores are the model\'s own uncalibrated predicted IoU for each candidate, not a "\n            "measured overlap and not a probability; the regression head is unclipped, so a value "\n            "may exceed 1.0"\n        ),\n        "sample_kind": sample_kind,\n        "n_masks": int(masks.shape[0]) if masks.ndim == 3 else 0,\n        "best_candidate": best,\n        "iou_scores_model_predicted": scores,\n        "mask_areas_px": [int(mask.sum()) for mask in masks] if masks.ndim == 3 else [],\n        "baselines": [],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n    if reference_mask is None:\n        return {\n            **base,\n            "metrics": [],\n            "verdict": "not-measurable",\n            "reason": "no ground-truth mask was supplied for the evaluated image",\n            "needs": (\n                "hand-labelled boolean masks for the prompted objects on your own images, scored with "\n                "mask_iou per object and averaged into a mean IoU over a held-out set; no labelled "\n                "mask set ships with this repository"\n            ),\n        }\n    reference = np.asarray(reference_mask)\n    return {\n        **base,\n        "metrics": [\n            {\n                "id": "mask_iou",\n                "candidate": index,\n                "value": mask_iou(masks[index], reference),\n                "selected": index == best,\n                "estimation": "one reference mask on a single scene, no dispersion estimate",\n            }\n            for index in range(masks.shape[0])\n        ],\n        "reference_area_px": int(reference.sum()),\n        "verdict": "sample-sanity",\n        "reason": (\n            "one reference mask on one tutorial sample; geometry sanity evidence, not a segmentation "\n            "benchmark"\n        ),\n        "needs": (\n            "a labelled mask set from the deployment domain for any mean-IoU or boundary-quality claim"\n        ),\n    }\n\n\ndef validate_segmentation_dataset(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n    """Validate paired images, prompts, and boolean masks for bounded adaptation."""\n    if isinstance(records, str | bytes) or not isinstance(records, Sequence):\n        raise TypeError("records must be a sequence of mappings")\n    if not 2 <= len(records) <= MAX_ADAPTATION_RECORDS:\n        raise ValueError(f"record count {len(records)} outside 2..{MAX_ADAPTATION_RECORDS}")\n\n    ids: list[str] = []\n    digest = hashlib.sha256()\n    positive_pixels = 0\n    for index, record in enumerate(records):\n        if not isinstance(record, Mapping):\n            raise TypeError(f"record {index} must be a mapping")\n        record_id = record.get("id")\n        if not isinstance(record_id, str) or not record_id.strip():\n            raise ValueError(f"record {index} id must be a non-empty string")\n        if record_id in ids:\n            raise ValueError(f"duplicate record id: {record_id}")\n        ids.append(record_id)\n\n        image = validate_image(record.get("image"))\n        mask = np.asarray(record.get("mask"))\n        if mask.dtype != np.bool_:\n            raise TypeError(f"record {record_id} mask must be boolean")\n        if mask.shape != (image.height, image.width):\n            raise ValueError(\n                f"record {record_id} mask shape {mask.shape} != image {(image.height, image.width)}"\n            )\n        area = int(mask.sum())\n        if area == 0 or area == mask.size:\n            raise ValueError(f"record {record_id} mask must contain foreground and background")\n        clean_points, clean_labels, clean_box = validate_prompts(\n            image.width,\n            image.height,\n            record.get("points"),\n            record.get("point_labels"),\n            record.get("box"),\n        )\n        if clean_points is not None and clean_labels is not None:\n            for (x, y), label in zip(clean_points, clean_labels, strict=True):\n                row = min(int(y), image.height - 1)\n                column = min(int(x), image.width - 1)\n                pixel_is_foreground = bool(mask[row, column])\n                if pixel_is_foreground != bool(label):\n                    raise ValueError(\n                        f"record {record_id} point ({x}, {y}) label {label} contradicts target mask"\n                    )\n        if clean_box is not None:\n            x0, y0 = (math.floor(value) for value in clean_box[:2])\n            x1, y1 = (math.ceil(value) for value in clean_box[2:])\n            if not bool(mask[y0:y1, x0:x1].any()):\n                raise ValueError(f"record {record_id} box does not overlap target mask foreground")\n        positive_pixels += area\n        digest.update(record_id.encode("utf-8"))\n        digest.update(\n            json.dumps(\n                {"image_shape": [image.height, image.width, 3], "mask_shape": list(mask.shape)},\n                sort_keys=True,\n                separators=(",", ":"),\n            ).encode("utf-8")\n        )\n        digest.update(np.asarray(image, dtype=np.uint8).tobytes())\n        digest.update(mask.tobytes())\n        digest.update(\n            json.dumps(\n                {"points": clean_points, "point_labels": clean_labels, "box": clean_box},\n                sort_keys=True,\n                separators=(",", ":"),\n            ).encode("utf-8")\n        )\n\n    return {\n        "records": len(records),\n        "unique_ids": len(ids),\n        "positive_pixels": positive_pixels,\n        "dataset_sha256": digest.hexdigest(),\n        "verdict": "accepted",\n    }\n\n\ndef _segmentation_record_content_sha256(record: Mapping[str, Any]) -> str:\n    """Fingerprint one validated image/mask sample without its ID or mutable prompts."""\n    image = validate_image(record["image"])\n    mask = np.asarray(record["mask"], dtype=np.bool_)\n    digest = hashlib.sha256()\n    digest.update(\n        json.dumps(\n            {"image_shape": [image.height, image.width, 3], "mask_shape": list(mask.shape)},\n            sort_keys=True,\n            separators=(",", ":"),\n        ).encode("utf-8")\n    )\n    digest.update(np.asarray(image, dtype=np.uint8).tobytes())\n    digest.update(mask.tobytes())\n    return digest.hexdigest()\n\n\n@dataclass\nclass SAM2SegmentationPipeline:\n    """Promptable image segmentation (points/box -> masks) over the pinned SAM 2.1 Hiera-Small checkpoint.\n\n    Image mode only: `Sam2Model` + `Sam2Processor`. Video tracking (`Sam2VideoModel`) is not exposed."""\n\n    _runner: Callable[..., tuple[np.ndarray, list[float]]]\n    device: str\n    model: Any | None = None\n    processor: Any | None = None\n    adaptation_config: dict[str, Any] = field(default_factory=dict)\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> SAM2SegmentationPipeline:\n        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n        if (root / MANIFEST_NAME).is_file():\n            stage_missing_files(root, allow_download=allow_download)\n            verify_snapshot(root)\n            source, kwargs = str(root), {"local_files_only": True}\n        elif allow_download:\n            source, kwargs = MODEL_ID, {}\n        else:\n            raise FileNotFoundError(\n                f"no verified snapshot at {root} and allow_download=False; "\n                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"\n            )\n        # Refuse invalid snapshots before importing model libraries.\n        import torch\n        from transformers import Sam2Model, Sam2Processor\n        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n        processor = Sam2Processor.from_pretrained(\n            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs\n        )\n        # config.json declares Sam2VideoModel; image-mode Sam2Model loads the same weights (upstream README).\n        model = Sam2Model.from_pretrained(source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs)\n        model = model.to(resolved_device).eval()\n\n        def runner(image, points, labels, box, multimask) -> tuple[np.ndarray, list[float]]:\n            prompt_kwargs: dict[str, Any] = {}\n            if points is not None:\n                prompt_kwargs["input_points"] = [[points]]\n                prompt_kwargs["input_labels"] = [[labels]]\n            if box is not None:\n                prompt_kwargs["input_boxes"] = [[box]]\n            inputs = processor(images=image, return_tensors="pt", **prompt_kwargs).to(resolved_device)\n            with torch.inference_mode():\n                outputs = model(**inputs, multimask_output=multimask)\n            masks = processor.post_process_masks(\n                outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), mask_threshold=MASK_THRESHOLD\n            )[0]\n            return masks[0].numpy().astype(np.bool_), [float(v) for v in outputs.iou_scores[0, 0].tolist()]\n\n        return cls(runner, resolved_device, model=model, processor=processor)\n\n    def freeze_for_adaptation(self) -> dict[str, int]:\n        """Freeze the base model and enable the first mask-token hypernetwork only."""\n        if self.model is None:\n            raise RuntimeError("cannot configure adaptation without an underlying torch model")\n        previous_config = dict(self.adaptation_config)\n        trainable = frozen = 0\n        for name, parameter in self.model.named_parameters():\n            parameter.requires_grad = name.startswith(TRAINABLE_PREFIXES)\n            if parameter.requires_grad:\n                trainable += parameter.numel()\n            else:\n                frozen += parameter.numel()\n        if trainable == 0:\n            raise RuntimeError("SAM2 adaptation selected no trainable parameters")\n        base_config = {\n            "method": ADAPTATION_METHOD,\n            "trainable_prefixes": list(TRAINABLE_PREFIXES),\n            "trainable_parameters": trainable,\n            "frozen_parameters": frozen,\n        }\n        if any(key in previous_config for key in _ADAPTATION_RUN_FIELDS):\n            _validate_completed_adaptation_metadata(previous_config)\n            previous_config.update(base_config)\n            self.adaptation_config = previous_config\n        else:\n            self.adaptation_config = base_config\n        return {"trainable_parameters": trainable, "frozen_parameters": frozen}\n\n    def finetune(\n        self,\n        train_records: Sequence[Mapping[str, Any]],\n        val_records: Sequence[Mapping[str, Any]] | None = None,\n        *,\n        epochs: int = 2,\n        learning_rate: float = 2e-5,\n        seed: int = 42,\n    ) -> list[dict[str, Any]]:\n        """Run bounded mask-hypernetwork fine-tuning with BCE plus soft-Dice loss."""\n        import random\n\n        import torch\n        import torch.nn.functional as F\n        from torch.optim import AdamW\n\n        if self.model is None or self.processor is None:\n            raise RuntimeError("cannot fine-tune without the underlying model and processor")\n        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:\n            raise ValueError("epochs must be an int in 1..20")\n        if not isinstance(learning_rate, int | float) or isinstance(learning_rate, bool):\n            raise TypeError("learning_rate must be numeric")\n        if not 0 < float(learning_rate) <= 1e-2:\n            raise ValueError("learning_rate must be in (0, 1e-2]")\n        if not isinstance(seed, int) or isinstance(seed, bool):\n            raise ValueError("seed must be an int")\n        train_manifest = validate_segmentation_dataset(train_records)\n        val_manifest = validate_segmentation_dataset(val_records) if val_records else None\n        adaptation_pixels = sum(\n            record["image"].width * record["image"].height for record in train_records\n        ) + sum(record["image"].width * record["image"].height for record in (val_records or ()))\n        if adaptation_pixels > MAX_ADAPTATION_PIXELS:\n            raise ValueError(\n                f"adaptation aggregate pixel count {adaptation_pixels} exceeds "\n                f"MAX_ADAPTATION_PIXELS {MAX_ADAPTATION_PIXELS}"\n            )\n        if val_records:\n            train_ids = {str(record["id"]) for record in train_records}\n            val_ids = {str(record["id"]) for record in val_records}\n            overlapping_ids = sorted(train_ids & val_ids)\n            if overlapping_ids:\n                raise ValueError(\n                    f"train and validation records overlap by id: {overlapping_ids[:5]}"\n                )\n            train_content = {\n                _segmentation_record_content_sha256(record) for record in train_records\n            }\n            val_content = {\n                _segmentation_record_content_sha256(record) for record in val_records\n            }\n            if train_content & val_content:\n                raise ValueError("train and validation records overlap by image/mask content")\n\n        entry_config = copy.deepcopy(self.adaptation_config)\n        entry_requires_grad = {\n            name: parameter.requires_grad for name, parameter in self.model.named_parameters()\n        }\n        entry_parameter_devices = {\n            name: parameter.device for name, parameter in self.model.named_parameters()\n        }\n        entry_parameter_grads = {\n            name: None if parameter.grad is None else parameter.grad.detach().clone()\n            for name, parameter in self.model.named_parameters()\n        }\n        entry_buffer_devices = {\n            name: buffer.device for name, buffer in self.model.named_buffers()\n        }\n        entry_training = self.model.training\n\n        def restore_entry_state() -> None:\n            self.adaptation_config = copy.deepcopy(entry_config)\n            with torch.no_grad():\n                for name, parameter in self.model.named_parameters():\n                    parameter.data = parameter.data.to(entry_parameter_devices[name])\n                    entry_grad = entry_parameter_grads[name]\n                    parameter.grad = (\n                        None\n                        if entry_grad is None\n                        else entry_grad.to(entry_parameter_devices[name]).clone()\n                    )\n                    parameter.requires_grad = entry_requires_grad[name]\n                for name, buffer in self.model.named_buffers():\n                    buffer.data = buffer.data.to(entry_buffer_devices[name])\n            self.model.train(entry_training)\n\n        device = torch.device(self.device)\n\n        def prepare(\n            record: Mapping[str, Any],\n        ) -> tuple[list[torch.Tensor], dict[str, torch.Tensor], torch.Tensor, torch.Tensor]:\n            prompt_kwargs: dict[str, Any] = {}\n            if record.get("points") is not None:\n                prompt_kwargs["input_points"] = [[record["points"]]]\n                prompt_kwargs["input_labels"] = [[record["point_labels"]]]\n            if record.get("box") is not None:\n                prompt_kwargs["input_boxes"] = [[record["box"]]]\n            batch = self.processor(\n                images=record["image"], return_tensors="pt", **prompt_kwargs\n            ).to(device)\n            with torch.inference_mode():\n                embeddings = [\n                    item.detach().cpu()\n                    for item in self.model.get_image_embeddings(batch["pixel_values"])\n                ]\n            prompts = {\n                key: batch[key].detach().cpu()\n                for key in ("input_points", "input_labels", "input_boxes")\n                if key in batch\n            }\n            target = torch.from_numpy(np.asarray(record["mask"], dtype=np.float32))\n            original_size = torch.tensor([[record["image"].height, record["image"].width]])\n            return embeddings, prompts, target, original_size\n\n        def score(\n            cached: Sequence[\n                tuple[list[torch.Tensor], dict[str, torch.Tensor], torch.Tensor, torch.Tensor]\n            ],\n        ) -> float:\n            values: list[float] = []\n            self.model.eval()\n            with torch.inference_mode():\n                for embeddings, prompts, target, original_size in cached:\n                    device_embeddings = [item.to(device) for item in embeddings]\n                    device_prompts = {key: value.to(device) for key, value in prompts.items()}\n                    device_target = target.to(device)\n                    outputs = self.model(\n                        image_embeddings=device_embeddings,\n                        multimask_output=False,\n                        **device_prompts,\n                    )\n                    public_mask = self.processor.post_process_masks(\n                        outputs.pred_masks.cpu(),\n                        original_size,\n                        mask_threshold=MASK_THRESHOLD,\n                    )[0][0, 0]\n                    values.append(mask_iou(public_mask.numpy(), device_target.cpu().numpy().astype(bool)))\n            return float(np.mean(values)) if values else 0.0\n\n        try:\n            torch.manual_seed(seed)\n            self.model.to(device).eval()\n            cached_train = [prepare(record) for record in train_records]\n            cached_val = [prepare(record) for record in val_records] if val_records else []\n            baseline_val_iou = score(cached_val) if cached_val else None\n        except BaseException:\n            restore_entry_state()\n            raise\n\n        try:\n            if not self.adaptation_config:\n                self.freeze_for_adaptation()\n            if any(\n                parameter.requires_grad and not name.startswith(TRAINABLE_PREFIXES)\n                for name, parameter in self.model.named_parameters()\n            ):\n                raise RuntimeError("parameters outside the declared SAM2 adapter surface are trainable")\n            trainable = {\n                name: parameter\n                for name, parameter in self.model.named_parameters()\n                if parameter.requires_grad\n            }\n            if not trainable:\n                raise RuntimeError("model has no trainable parameters")\n            before = {\n                name: parameter.detach().cpu().clone() for name, parameter in trainable.items()\n            }\n            optimizer = AdamW(list(trainable.values()), lr=float(learning_rate))\n            previous_config = copy.deepcopy(self.adaptation_config)\n            prior_runs = previous_config.get("training_runs")\n            if prior_runs is None:\n                prior_runs = (\n                    [_adaptation_run_snapshot(previous_config)]\n                    if _is_positive_finite_number(previous_config.get("weight_delta_l2"))\n                    and isinstance(previous_config.get("history"), list)\n                    and previous_config["history"]\n                    else []\n                )\n            elif not isinstance(prior_runs, list) or not all(\n                isinstance(run, Mapping) for run in prior_runs\n            ):\n                raise RuntimeError("adaptation training_runs metadata is malformed")\n            else:\n                prior_runs = [json.loads(json.dumps(run)) for run in prior_runs]\n            for key in (\n                "epochs",\n                "learning_rate",\n                "batch_size",\n                "seed",\n                "loss",\n                "train_manifest",\n                "validation_manifest",\n                "baseline_validation_mask_iou",\n                "history",\n                "weight_delta_l2",\n            ):\n                self.adaptation_config.pop(key, None)\n        except BaseException:\n            restore_entry_state()\n            raise\n\n        history: list[dict[str, Any]] = []\n        try:\n            for epoch in range(1, epochs + 1):\n                order = list(range(len(cached_train)))\n                random.Random(seed + epoch * 17).shuffle(order)\n                total_loss = 0.0\n                for index in order:\n                    embeddings, prompts, target, _original_size = cached_train[index]\n                    device_embeddings = [item.to(device) for item in embeddings]\n                    device_prompts = {key: value.to(device) for key, value in prompts.items()}\n                    device_target = target.to(device)\n                    optimizer.zero_grad(set_to_none=True)\n                    logits = self.model(\n                        image_embeddings=device_embeddings,\n                        multimask_output=False,\n                        **device_prompts,\n                    ).pred_masks[0, 0, 0]\n                    target_low = (\n                        F.interpolate(\n                            device_target[None, None], size=logits.shape, mode="area"\n                        )[0, 0]\n                        > 0\n                    ).to(logits.dtype)\n                    probabilities = logits.sigmoid()\n                    bce = F.binary_cross_entropy_with_logits(logits, target_low)\n                    dice = 1.0 - (2.0 * (probabilities * target_low).sum() + 1.0) / (\n                        probabilities.sum() + target_low.sum() + 1.0\n                    )\n                    loss = bce + dice\n                    if not bool(torch.isfinite(loss)):\n                        raise RuntimeError("fine-tuning produced a non-finite loss")\n                    loss.backward()\n                    optimizer.step()\n                    if any(\n                        not bool(torch.isfinite(parameter).all())\n                        for parameter in trainable.values()\n                    ):\n                        raise RuntimeError("fine-tuning produced non-finite adapter weights")\n                    total_loss += float(loss.item())\n                epoch_data: dict[str, Any] = {\n                    "epoch": epoch,\n                    "train_loss": round(total_loss / len(cached_train), 6),\n                    "optimizer_steps": len(cached_train),\n                }\n                if cached_val:\n                    epoch_data["val_mask_iou"] = round(score(cached_val), 6)\n                history.append(epoch_data)\n\n            delta_sq = 0.0\n            for name, parameter in trainable.items():\n                delta_sq += float(\n                    torch.sum((parameter.detach().cpu() - before[name]) ** 2).item()\n                )\n            weight_delta_l2 = delta_sq**0.5\n            if not math.isfinite(weight_delta_l2):\n                raise RuntimeError("fine-tuning produced a non-finite weight delta")\n            if weight_delta_l2 == 0.0:\n                raise RuntimeError("fine-tuning completed without changing adapter weights")\n            self.model.eval()\n            completed_run = {\n                "epochs": epochs,\n                "learning_rate": float(learning_rate),\n                "batch_size": 1,\n                "seed": seed,\n                "loss": "binary-cross-entropy-plus-soft-dice",\n                "train_manifest": train_manifest,\n                "validation_manifest": val_manifest,\n                "baseline_validation_mask_iou": baseline_val_iou,\n                "history": history,\n                "weight_delta_l2": weight_delta_l2,\n            }\n            stored_run = json.loads(json.dumps(completed_run))\n            self.adaptation_config.update(stored_run)\n            self.adaptation_config["training_runs"] = [\n                *prior_runs,\n                json.loads(json.dumps(stored_run)),\n            ]\n        except BaseException:\n            with torch.no_grad():\n                for name, parameter in trainable.items():\n                    parameter.copy_(before[name].to(device=parameter.device, dtype=parameter.dtype))\n            restore_entry_state()\n            raise\n        return json.loads(json.dumps(history))\n\n    def evaluate_adaptation(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n        """Evaluate adapted masks against ground truth and a prompt-box baseline."""\n        validate_segmentation_dataset(records)\n        model_ious: list[float] = []\n        box_ious: list[float] = []\n        box_model_ious: list[float] = []\n        for record in records:\n            result = self.segment(\n                record["image"],\n                points=record.get("points"),\n                point_labels=record.get("point_labels"),\n                box=record.get("box"),\n                multimask=False,\n            )\n            target = np.asarray(record["mask"], dtype=np.bool_)\n            model_iou = mask_iou(result["masks"][0], target)\n            model_ious.append(model_iou)\n            box = record.get("box")\n            if box is not None:\n                baseline = np.zeros_like(target)\n                x0, y0 = (math.floor(float(value)) for value in box[:2])\n                x1, y1 = (math.ceil(float(value)) for value in box[2:])\n                baseline[y0:y1, x0:x1] = True\n                box_ious.append(mask_iou(baseline, target))\n                box_model_ious.append(model_iou)\n        box_baseline_mean_iou = float(np.mean(box_ious)) if box_ious else None\n        box_prompt_model_mean_iou = float(np.mean(box_model_ious)) if box_model_ious else None\n        return {\n            "records": len(records),\n            "mean_mask_iou": float(np.mean(model_ious)),\n            "per_record_mask_iou": model_ious,\n            "box_baseline_records": len(box_ious),\n            "box_prompt_model_mean_iou": box_prompt_model_mean_iou,\n            "box_baseline_mean_iou": box_baseline_mean_iou,\n            "delta_over_box_baseline": (\n                None\n                if box_baseline_mean_iou is None\n                else float(box_prompt_model_mean_iou - box_baseline_mean_iou)\n            ),\n        }\n\n    def save_artifact(self, output_dir: str | Path, *, producer_revision: str) -> Path:\n        """Write a safe mask-hypernetwork adapter plus a closed integrity manifest."""\n        from safetensors.torch import save_file\n\n        if self.model is None:\n            raise RuntimeError("artifact export requires an underlying model")\n        try:\n            _validate_completed_adaptation_metadata(self.adaptation_config)\n        except ValueError as exc:\n            raise RuntimeError("artifact export requires completed fine-tuning metadata") from exc\n        if len(producer_revision) != 40 or any(ch not in "0123456789abcdef" for ch in producer_revision):\n            raise ValueError("producer_revision must be a lowercase 40-hex Git commit")\n        state = {}\n        for name, tensor in self.model.state_dict().items():\n            if not name.startswith(TRAINABLE_PREFIXES):\n                continue\n            if not bool(tensor.isfinite().all()):\n                raise RuntimeError(f"cannot export non-finite adapter tensor: {name}")\n            state[name] = tensor.detach().cpu().contiguous()\n        if not state:\n            raise RuntimeError("no adapter tensors selected for export")\n        root = Path(output_dir)\n        root.mkdir(parents=True, exist_ok=True)\n        if any(root.iterdir()):\n            raise FileExistsError(f"artifact directory is not empty: {root}")\n        weights_path = root / ARTIFACT_WEIGHTS_NAME\n        save_file(state, str(weights_path))\n        manifest = {\n            "artifactSpec": "1.0",\n            "format": ARTIFACT_FORMAT,\n            "formatVersion": ARTIFACT_FORMAT_VERSION,\n            "artifactClass": "ADAPTER",\n            "artifactKind": "sam2-mask-hypernetwork-adapter",\n            "producer": {"pipelineId": "sam2-segmentation-pipeline", "revision": producer_revision},\n            "createdAtUtc": datetime.now(UTC).isoformat(),\n            "baseModel": {"id": MODEL_ID, "revision": MODEL_REVISION},\n            "files": [\n                {\n                    "path": ARTIFACT_WEIGHTS_NAME,\n                    "bytes": weights_path.stat().st_size,\n                    "sha256": _sha256(weights_path),\n                }\n            ],\n            "adaptation": copy.deepcopy(self.adaptation_config),\n            "trainablePrefixes": list(TRAINABLE_PREFIXES),\n            "retainedData": {"containsTrainingRecords": False, "containsSupportRecords": False},\n            "serialization": "safetensors",\n        }\n        (root / ARTIFACT_MANIFEST_NAME).write_text(\n            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"\n        )\n        return root\n\n    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n        """Verify and load a SAM2 adapter without code-capable deserialization."""\n        from safetensors.torch import load_file\n\n        if self.model is None:\n            raise RuntimeError("cannot load an artifact without an underlying torch model")\n        root = Path(artifact_dir)\n        manifest_path = root / ARTIFACT_MANIFEST_NAME\n        if not manifest_path.is_file():\n            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")\n        expected_files = {ARTIFACT_MANIFEST_NAME, ARTIFACT_WEIGHTS_NAME}\n        actual_entries = {path.name for path in root.iterdir()}\n        if actual_entries != expected_files:\n            raise ValueError(\n                f"artifact directory must contain exactly {sorted(expected_files)}, "\n                f"found {sorted(actual_entries)}"\n            )\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if manifest.get("format") != ARTIFACT_FORMAT:\n            raise ValueError(f"unrecognized artifact format: {manifest.get(\'format\')}")\n        if manifest.get("formatVersion") != ARTIFACT_FORMAT_VERSION:\n            raise ValueError(f"unsupported artifact formatVersion: {manifest.get(\'formatVersion\')}")\n        if manifest.get("baseModel") != {"id": MODEL_ID, "revision": MODEL_REVISION}:\n            raise ValueError("artifact base model identity is incompatible")\n        if manifest.get("trainablePrefixes") != list(TRAINABLE_PREFIXES):\n            raise ValueError("artifact trainable prefixes do not match this pipeline")\n        producer = manifest.get("producer")\n        if (\n            not isinstance(producer, Mapping)\n            or set(producer) != {"pipelineId", "revision"}\n            or producer["pipelineId"] != "sam2-segmentation-pipeline"\n            or not isinstance(producer["revision"], str)\n            or len(producer["revision"]) != 40\n            or any(character not in "0123456789abcdef" for character in producer["revision"])\n        ):\n            raise ValueError("artifact producer identity is invalid")\n        files = manifest.get("files")\n        if not isinstance(files, list) or len(files) != 1:\n            raise ValueError("artifact manifest must inventory exactly one weights file")\n        entry = files[0]\n        if entry.get("path") != ARTIFACT_WEIGHTS_NAME:\n            raise ValueError("artifact manifest names an unexpected weights path")\n        weights_path = root / ARTIFACT_WEIGHTS_NAME\n        if not weights_path.is_file():\n            raise FileNotFoundError(f"artifact weights not found: {weights_path}")\n        if weights_path.stat().st_size != entry.get("bytes") or _sha256(weights_path) != entry.get("sha256"):\n            raise ValueError("artifact weights failed size or SHA-256 verification")\n        adaptation = manifest.get("adaptation")\n        _validate_completed_adaptation_metadata(adaptation)\n        state = load_file(str(weights_path), device=self.device)\n        expected = {\n            name for name in self.model.state_dict() if name.startswith(TRAINABLE_PREFIXES)\n        }\n        if set(state) != expected:\n            raise ValueError("artifact tensor inventory does not match the declared adapter surface")\n        destination = self.model.state_dict()\n        for name, tensor in state.items():\n            expected_tensor = destination[name]\n            if tensor.shape != expected_tensor.shape or tensor.dtype != expected_tensor.dtype:\n                raise ValueError(\n                    f"artifact tensor {name} has shape/dtype {tuple(tensor.shape)}/{tensor.dtype}; "\n                    f"expected {tuple(expected_tensor.shape)}/{expected_tensor.dtype}"\n                )\n            if not bool(tensor.isfinite().all()):\n                raise ValueError(f"artifact tensor {name} contains non-finite values")\n        self.model.load_state_dict(state, strict=False)\n        self.model.to(self.device).eval()\n        # A fresh base model starts fully trainable. Reapply the declared adapter surface so a\n        # verified artifact can be continued with ``finetune`` without exposing base parameters.\n        self.freeze_for_adaptation()\n        loaded_adaptation = copy.deepcopy(adaptation)\n        prior_artifacts = loaded_adaptation.get("artifact_lineage", [])\n        loaded_adaptation["artifact_lineage"] = [\n            *json.loads(json.dumps(prior_artifacts)),\n            {\n                "manifest_sha256": _sha256(manifest_path),\n                "weights_sha256": entry["sha256"],\n                "producer": copy.deepcopy(producer),\n            },\n        ]\n        self.adaptation_config = loaded_adaptation\n        return manifest\n\n    @classmethod\n    def from_artifact(\n        cls,\n        artifact_dir: str | Path,\n        *,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> SAM2SegmentationPipeline:\n        """Construct a fresh pinned base model and attach a verified adapter."""\n        pipeline = cls.from_pretrained(\n            device=device, weights_dir=weights_dir, allow_download=allow_download\n        )\n        pipeline.load_artifact(artifact_dir)\n        return pipeline\n\n    def segment(\n        self,\n        image: Image.Image,\n        *,\n        points: Sequence[Sequence[float]] | None = None,\n        point_labels: Sequence[int] | None = None,\n        box: Sequence[float] | None = None,\n        multimask: bool | None = None,\n    ) -> dict[str, Any]:\n        """Segment one object, defaulting adapted pipelines to their trained single-mask head."""\n        resolved_multimask = not bool(self.adaptation_config) if multimask is None else multimask\n        rgb, clean_points, clean_labels, clean_box = _check_inputs(\n            image, points, point_labels, box, resolved_multimask\n        )\n        if self.adaptation_config and resolved_multimask:\n            raise ValueError(\n                "adapted pipelines require multimask=False because the adapter trains only "\n                "the single-mask output hypernetwork"\n            )\n        masks, iou_scores = self._runner(\n            rgb, clean_points, clean_labels, clean_box, resolved_multimask\n        )\n        masks = np.asarray(masks)\n        expected = (NUM_MULTIMASK_OUTPUTS if resolved_multimask else 1, rgb.height, rgb.width)\n        if masks.dtype != np.bool_ or masks.shape != expected or len(iou_scores) != expected[0]:\n            raise RuntimeError(f"backend returned {masks.shape} {masks.dtype}, {len(iou_scores)} scores")\n        return {\n            "masks": masks,\n            "iou_scores": [float(v) for v in iou_scores],\n            "multimask": resolved_multimask,\n            "points": clean_points,\n            "point_labels": clean_labels,\n            "box": clean_box,\n            "width": rgb.width,\n            "height": rgb.height,\n            "model_id": MODEL_ID,\n            "model_revision": MODEL_REVISION,\n        }\n', 'tutorial_stages.py': '"""Stage runner for the standalone SAM 2.1 segmentation E2E tutorial (NOTEBOOK_SPEC 2.2 §25.13 isolated-environment pattern).\n\nThe tutorial notebook carries this file verbatim (as ``tutorial_stages.py`` in its run directory, beside the carried\npackage under ``src/``) and runs every stage with the interpreter of an isolated, hash-locked environment::\n\n    python -u tutorial_stages.py --root RUN_DIR --outputs OUTPUTS --weights WEIGHTS --stage data --options \'{...}\'\n\nNothing is installed into the notebook kernel. Each stage is a separate process and starts from files only: the\nverified snapshot under ``--weights``, the records of earlier stages under ``RUN_DIR/state`` (each record as an RGB PNG,\na mask PNG and its prompts in JSON), and the learner-facing exports under ``--outputs``. The fine-tune always starts\nfrom the pinned base, and the adapted model reaches later stages only through the exported adapter, which the\n``evaluate`` and ``unseen`` stages attach to a fresh base in their own processes. On failure a stage writes\n``RUN_DIR/state/<stage>.error.json``, which the notebook re-raises in the kernel.\n\nStages: weights → data → validate → baseline → adapt → evaluate → unseen → result, plus the optional ``activity``. The\n``data`` and ``validate`` stages import no model library, so CI exercises them directly. No stage asserts a quality\nlevel; only a broken reload contract (a fresh base plus the adapter not reproducing the adapting process) stops the run.\n"""\n# ruff: noqa: E501  -- the printed dictionaries are the learner-facing output; they are kept on one line each\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport io\nimport json\nimport math\nimport random\nimport shutil\nimport sys\nimport traceback\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any\n\nSTEM = "sam2_segmentation"\nPACKAGE = "sam2_segmentation_pipeline"\nSEED = 42\nGENERATED_RECORDS = 24\nGENERATED_TRAIN = 18  # generated-00..17 train, generated-18..23 held out (the recorded split)\nHELD_OUT_FRACTION = 0.25\nMIN_RECORDS = 8\nARTIFACT_DIRNAME = "sam2-mask-hypernetwork-adapter-v1"\nRELOAD_RTOL, RELOAD_ATOL = 1e-5, 1e-6\nSATURATED_IOU = 0.99  # a held-out mean IoU at or above this leaves no room to show an adaptation effect\nBYOD_ZIP_MAX_BYTES = 256 * 1024 * 1024\nBYOD_MAX_MEMBERS = 130\nBYOD_MAX_UNCOMPRESSED = 512 * 1024 * 1024\nBYOD_MAX_TOTAL_PIXELS = 32 * 1024 * 1024\nCONDITIONS = {\n    "point+box": "the shape\'s exact bounding box and one interior point (the recorded condition)",\n    "point-only": "one interior point, no box (the model must find the extent itself; on these high-contrast scenes it is at ceiling too)",\n}\n\n\nclass Run:\n    """Paths of one run: carried sources and state under ``root``; learner-facing files under ``outputs``."""\n\n    def __init__(self, root: Path, outputs: Path, weights: Path, options: dict[str, Any]) -> None:\n        self.root = root\n        self.out = outputs\n        self.weights = weights\n        self.options = options\n        self.state = root / "state"\n        self.out.mkdir(parents=True, exist_ok=True)\n        self.state.mkdir(parents=True, exist_ok=True)\n\n    def write_state(self, name: str, value: Any) -> Path:\n        path = self.state / name\n        path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n        return path\n\n    def read_state(self, name: str, needed_by: str) -> Any:\n        path = self.state / name\n        if not path.is_file():\n            raise RuntimeError(f"{name} is missing: run the stage that writes it before \'{needed_by}\' (run the notebook in order from Section 4)")\n        return json.loads(path.read_text(encoding="utf-8"))\n\n    def write_output(self, name: str, value: Any) -> Path:\n        path = self.out / name\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n        return path\n\n\ndef package(root: Path):\n    src = str(root / "src")\n    if src not in sys.path:\n        sys.path.insert(0, src)\n    return importlib.import_module(f"{PACKAGE}.pipeline")  # the module: it also defines MANIFEST_NAME and ARTIFACT_MANIFEST_NAME\n\n\ndef snapshot_dir(run: Run, P) -> Path:\n    return run.weights / P.MODEL_KEY\n\n\ndef rounded(value: Any, digits: int = 6) -> Any:\n    if isinstance(value, float):\n        return round(value, digits) if math.isfinite(value) else value\n    if isinstance(value, dict):\n        return {k: rounded(v, digits) for k, v in value.items()}\n    if isinstance(value, list | tuple):\n        return [rounded(v, digits) for v in value]\n    return value\n\n\ndef source_record(run: Run) -> dict[str, Any]:\n    path = run.root / "source.json"\n    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}\n\n\n# --------------------------------------------------------------------------------------------------\n# records (data and validate import no model library)\n# --------------------------------------------------------------------------------------------------\n\n\ndef record_from_pair(record_id: str, image, mask_image) -> dict[str, Any]:\n    import numpy as np\n\n    image = image.convert("RGB")\n    mask = np.asarray(mask_image.convert("L")) > 127\n    ys, xs = np.where(mask)\n    if len(xs) == 0:\n        raise ValueError(f"{record_id}: the mask is empty (no pixel above 127); every mask needs foreground")\n    x0, x1, y0, y1 = int(xs.min()), int(xs.max()) + 1, int(ys.min()), int(ys.max()) + 1\n    point = [float(np.median(xs)), float(np.median(ys))]\n    if not mask[int(point[1]), int(point[0])]:\n        point = [float(xs[0]), float(ys[0])]\n    return {"id": record_id, "image": image, "mask": mask, "points": [point], "point_labels": [1], "box": [float(x0), float(y0), float(x1), float(y1)]}\n\n\ndef generated_records(start: int = 0, count: int = GENERATED_RECORDS) -> list[dict[str, Any]]:\n    """The deterministic tutorial scenes: one bright rounded rectangle or ellipse and one distractor on a flat dark\n    background (high contrast on purpose: see Section 6 for what that means for the comparison)."""\n    import numpy as np\n    from PIL import Image, ImageDraw\n\n    records = []\n    for index in range(start, start + count):\n        rng = np.random.default_rng(8100 + index)\n        image = Image.new("RGB", (256, 256), tuple(int(v) for v in rng.integers(35, 95, 3)))\n        draw = ImageDraw.Draw(image)\n        mask_image = Image.new("L", image.size, 0)\n        mask_draw = ImageDraw.Draw(mask_image)\n        x0, y0 = int(rng.integers(28, 105)), int(rng.integers(28, 105))\n        width, height = int(rng.integers(70, 125)), int(rng.integers(65, 120))\n        box = [x0, y0, min(244, x0 + width), min(244, y0 + height)]\n        colour = tuple(int(v) for v in rng.integers(145, 245, 3))\n        if index % 2:\n            draw.ellipse(box, fill=colour)\n            mask_draw.ellipse(box, fill=255)\n        else:\n            radius = 18 + index % 12\n            draw.rounded_rectangle(box, radius=radius, fill=colour)\n            mask_draw.rounded_rectangle(box, radius=radius, fill=255)\n        distractor = [int(rng.integers(5, 55)), int(rng.integers(175, 215)), int(rng.integers(70, 120)), int(rng.integers(225, 250))]\n        draw.rectangle(distractor, fill=tuple(int(v) for v in rng.integers(90, 180, 3)))\n        records.append(record_from_pair(f"generated-{index:02d}", image, mask_image))\n    return records\n\n\ndef records_from_zip(P, blob: bytes, name: str) -> list[dict[str, Any]]:\n    """Read paired ``images/<stem>.(png|jpg|jpeg)`` and ``masks/<stem>.png`` members (optionally under one wrapping\n    folder) without extracting; every refusal names the archive and the rule (SAM-m1)."""\n    from PIL import Image\n\n    if len(blob) > BYOD_ZIP_MAX_BYTES:\n        raise ValueError(f"{name}: {len(blob)} bytes exceeds the 256 MiB BYOD ceiling")\n    try:\n        archive = zipfile.ZipFile(io.BytesIO(blob))\n    except zipfile.BadZipFile:\n        raise ValueError(f"{name}: not a ZIP archive") from None\n    with archive:\n        infos = archive.infolist()\n        if len(infos) > BYOD_MAX_MEMBERS:\n            raise ValueError(f"{name}: {len(infos)} members exceeds the {BYOD_MAX_MEMBERS}-member BYOD ceiling (at most 64 image/mask pairs)")\n        if sum(i.file_size for i in infos) > BYOD_MAX_UNCOMPRESSED:\n            raise ValueError(f"{name}: the members expand to more than 512 MiB; refusing to read it")\n        if any(i.flag_bits & 1 for i in infos):\n            raise ValueError(f"{name}: encrypted members are not supported")\n        names = [i.filename.replace("\\\\", "/") for i in infos if not i.is_dir()]\n        unsafe = [n for n in names if n.startswith("/") or ".." in n.split("/")]\n        if unsafe:\n            raise ValueError(f"{name}: unsafe member paths {unsafe[:3]} (absolute or containing \'..\'); refusing the archive")\n        prefix = ""\n        if not any(n.startswith(("images/", "masks/")) for n in names):\n            tops = sorted({n.split("/")[0] for n in names if "/" in n})\n            wrapped = [t for t in tops if any(n.startswith(f"{t}/images/") for n in names) and any(n.startswith(f"{t}/masks/") for n in names)]\n            if len(wrapped) == 1:\n                prefix = wrapped[0] + "/"\n                print({"byod_wrapping_folder": wrapped[0], "note": "pairs read from inside this single wrapping folder"})\n            else:\n                raise ValueError(f"{name}: expected images/ and masks/ at the top level or inside one wrapping folder; top-level folders found: {tops or \'none\'}")\n        originals = {i.filename.replace("\\\\", "/"): i.filename for i in infos}\n        images: dict[str, str] = {}\n        masks: dict[str, str] = {}\n        non_png_masks = []\n        for n in names:\n            if not n.startswith(prefix):\n                continue\n            rel = n[len(prefix):]\n            stem, _, ext = rel.split("/")[-1].rpartition(".")\n            if rel.startswith("images/") and ext.lower() in ("png", "jpg", "jpeg"):\n                if stem in images:\n                    raise ValueError(f"{name}: duplicate image stem {stem!r}")\n                images[stem] = originals[n]\n            elif rel.startswith("masks/"):\n                if ext.lower() != "png":\n                    non_png_masks.append(rel)\n                    continue\n                if stem in masks:\n                    raise ValueError(f"{name}: duplicate mask stem {stem!r}")\n                masks[stem] = originals[n]\n        if non_png_masks:\n            raise ValueError(f"{name}: masks must be PNG (`masks/<stem>.png`, lossless, white = foreground); found {non_png_masks[:3]}")\n        if set(images) != set(masks):\n            raise ValueError(f"{name}: every image needs a mask with the same stem; images without a mask: {sorted(set(images) - set(masks))[:5]}, masks without an image: {sorted(set(masks) - set(images))[:5]}")\n        if len(images) < MIN_RECORDS:\n            raise ValueError(f"{name}: found {len(images)} image/mask pair(s); the E2E path needs at least {MIN_RECORDS} (6 to train, 2 to hold out)")\n        records, total = [], 0\n        for stem in sorted(images):\n            image = Image.open(io.BytesIO(archive.read(images[stem])))\n            mask = Image.open(io.BytesIO(archive.read(masks[stem])))\n            if min(image.size) < P.MIN_IMAGE_SIDE or max(image.size) > P.MAX_IMAGE_SIDE or image.size != mask.size:\n                raise ValueError(f"{name}: {stem}: image {image.size} and mask {mask.size} must match and lie within {P.MIN_IMAGE_SIDE}..{P.MAX_IMAGE_SIDE} px")\n            total += image.width * image.height\n            if total > BYOD_MAX_TOTAL_PIXELS:\n                raise ValueError(f"{name}: the images exceed the 32-megapixel total decoded-pixel ceiling")\n            image.load()\n            mask.load()\n            records.append(record_from_pair(stem, image, mask))\n        return records\n\n\ndef group_split(records: list[dict[str, Any]], seed: int, separator: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]], str]:\n    """Seeded, group-aware held-out split for BYOD (SAM-m1): records sharing a group (the stem up to ``separator``) stay\n    on one side; without a separator every record is its own group (independent images assumed)."""\n    groups: dict[str, list[dict[str, Any]]] = {}\n    for r in records:\n        key = r["id"].split(separator)[0] if separator else r["id"]\n        groups.setdefault(key, []).append(r)\n    order = sorted(groups)\n    random.Random(seed).shuffle(order)\n    target = max(2, round(len(records) * HELD_OUT_FRACTION))\n    held: list[dict[str, Any]] = []\n    for key in order:\n        if len(held) >= target:\n            break\n        if len(records) - len(held) - len(groups[key]) < 2:\n            continue\n        held.extend(groups[key])\n    held_ids = {r["id"] for r in held}\n    train = [r for r in records if r["id"] not in held_ids]\n    if len(held) < 2 or len(train) < 2:\n        raise ValueError(f"the groups cannot be split into at least 2 training and 2 held-out records ({len(groups)} groups); add images or check GROUP_SEPARATOR={separator!r}")\n    note = f"seeded (seed {seed}) shuffle of {len(groups)} group(s) by the stem prefix before {separator!r}; a group never straddles the split" if separator else f"seeded (seed {seed}) shuffle of {len(records)} records, each its own group (independent images assumed: set GROUP_SEPARATOR if frames or crops share a scene)"\n    return train, held, note\n\n\ndef save_records(run: Run, records: list[dict[str, Any]], split: str, rows: list[dict[str, Any]]) -> None:\n    from PIL import Image\n\n    folder = run.state / "records"\n    folder.mkdir(exist_ok=True)\n    for r in records:\n        index = len(rows)\n        r["image"].save(folder / f"{index:03d}-image.png")\n        Image.fromarray(r["mask"].astype("uint8") * 255).save(folder / f"{index:03d}-mask.png")\n        rows.append({"id": r["id"], "index": index, "split": split, "points": r["points"], "point_labels": r["point_labels"], "box": r["box"]})\n\n\ndef load_records(run: Run, split: str, needed_by: str) -> list[dict[str, Any]]:\n    import numpy as np\n    from PIL import Image\n\n    rows = run.read_state("records.json", needed_by)\n    out = []\n    for row in rows:\n        if row["split"] != split:\n            continue\n        image = Image.open(run.state / "records" / f"{row[\'index\']:03d}-image.png").convert("RGB")\n        mask = np.asarray(Image.open(run.state / "records" / f"{row[\'index\']:03d}-mask.png").convert("L")) > 127\n        out.append({"id": row["id"], "image": image, "mask": mask, "points": row["points"], "point_labels": row["point_labels"], "box": row["box"]})\n    return out\n\n\ndef condition(records: list[dict[str, Any]], name: str) -> list[dict[str, Any]]:\n    return records if name == "point+box" else [{**r, "box": None} for r in records]\n\n\n# --------------------------------------------------------------------------------------------------\n# weights\n# --------------------------------------------------------------------------------------------------\n\n\ndef stage_weights(run: Run) -> None:\n    P = package(run.root)\n    snapshot = snapshot_dir(run, P)\n    snapshot.mkdir(parents=True, exist_ok=True)\n    carried = run.root / "weights" / P.MODEL_KEY / P.MANIFEST_NAME\n    manifest = json.loads(carried.read_text(encoding="utf-8"))\n    if (manifest["modelId"], manifest["revision"]) != (P.MODEL_ID, P.MODEL_REVISION):\n        raise RuntimeError("the carried manifest does not name the identity carried by the package; regenerate the notebook")\n    shutil.copyfile(carried, snapshot / P.MANIFEST_NAME)\n    print({"model_id": P.MODEL_ID, "revision": P.MODEL_REVISION, "license": P.MODEL_LICENSE, "files": len(manifest["files"]), "total_bytes": manifest["totalBytes"]})\n    fetched = P.stage_missing_files(snapshot, allow_download=True)\n    print({"weights_dir": str(snapshot), "fetched": fetched})\n    verified = P.verify_snapshot(snapshot)\n    print({"verified_files": verified["files"], "revision": verified["revision"], "total_bytes": verified.get("total_bytes")})\n    run.write_state("weights.json", {"snapshot": str(snapshot), "fetched": fetched})\n\n\n# --------------------------------------------------------------------------------------------------\n# data and validation (SAM-m1)\n# --------------------------------------------------------------------------------------------------\n\n\ndef find_byod_zip(opts: dict[str, Any]) -> Path:\n    raw = str(opts.get("byod_zip_path") or "").strip()\n    if raw:\n        path = Path(raw).expanduser()\n        if not path.is_file():\n            raise FileNotFoundError(f"BYOD_ZIP_PATH {raw!r} is not a file in this runtime: upload the ZIP or correct the path.")\n        return path\n    kaggle = Path("/kaggle/input")\n    if kaggle.is_dir():\n        candidates = sorted(kaggle.rglob("*.zip"))\n        if len(candidates) != 1:\n            raise ValueError(f"BYOD_ZIP_PATH is empty and /kaggle/input holds {len(candidates)} ZIP files, not exactly one: set BYOD_ZIP_PATH explicitly.")\n        print({"byod_zip": str(candidates[0]), "note": "the one ZIP attached under /kaggle/input"})\n        return candidates[0]\n    raise FileNotFoundError("BYOD_ZIP_PATH is empty: set it to your ZIP (or, in Colab, leave it empty to open the upload dialog).")\n\n\ndef stage_data(run: Run) -> None:\n    P = package(run.root)\n    opts = run.options\n    use_byod = bool(opts.get("use_byod", False))\n    seed = int(opts.get("seed", SEED))\n    separator = str(opts.get("group_separator") or "")\n    for path in sorted(run.out.glob(f"{STEM}_*")):\n        if path.is_file():\n            path.unlink()\n    for name in ("records.json", "data.json", "baseline.json", "adapt.json", "evaluate.json", "unseen.json"):\n        (run.state / name).unlink(missing_ok=True)\n    shutil.rmtree(run.state / "records", ignore_errors=True)\n    if use_byod:\n        path = find_byod_zip(opts)\n        if path.stat().st_size > BYOD_ZIP_MAX_BYTES:\n            raise ValueError(f"{path.name}: {path.stat().st_size} bytes exceeds the 256 MiB BYOD ceiling")\n        records = records_from_zip(P, path.read_bytes(), path.name)\n        train, held, split_note = group_split(records, seed, separator)\n        unseen = [held[0]]\n        unseen_source = f"held-out BYOD record {held[0][\'id\']!r} (also scored in Section 8)"\n        kind, origin = "BYOD", path.name\n    else:\n        records = generated_records()\n        train, held = records[:GENERATED_TRAIN], records[GENERATED_TRAIN:]\n        split_note = "fixed generated split: generated-00..17 train, generated-18..23 held out (the recorded split)"\n        unseen = generated_records(start=GENERATED_RECORDS, count=1)\n        unseen_source = f"a separately generated scene ({unseen[0][\'id\']}) outside the train and held-out index range"\n        kind, origin = "generated", "deterministic generated shape scenes (seeds 8100..8123)"\n    rows: list[dict[str, Any]] = []\n    save_records(run, train, "train", rows)\n    save_records(run, held, "held_out", rows)\n    save_records(run, unseen, "unseen", rows)\n    run.write_state("records.json", rows)\n    run.write_state("data.json", {"kind": kind, "origin": origin, "seed": seed, "split": split_note, "unseen_source": unseen_source, "unseen_id": unseen[0]["id"]})\n    print({"dataset_kind": kind, "origin": origin, "records": len(records), "train": len(train), "held_out": len(held)})\n    print({"split": split_note})\n    print({"unseen": unseen_source})\n\n\ndef stage_validate(run: Run) -> None:\n    P = package(run.root)\n    data = run.read_state("data.json", "validate")\n    train = load_records(run, "train", "validate")\n    held = load_records(run, "held_out", "validate")\n    train_manifest = P.validate_segmentation_dataset(train)\n    held_manifest = P.validate_segmentation_dataset(held)\n    overlap = sorted({r["id"] for r in train} & {r["id"] for r in held})\n    if overlap:\n        raise ValueError(f"train/held-out leakage: ids {overlap} are on both sides")\n    manifest = {"kind": data["kind"], "split": data["split"], "train": train_manifest, "validation": held_manifest, "overlap_ids": overlap}\n    run.write_output(f"{STEM}_dataset_manifest.json", manifest)\n    print(json.dumps({k: v for k, v in manifest.items() if k not in ("train", "validation")}, indent=2))\n    print({"train_records": train_manifest.get("records"), "held_out_records": held_manifest.get("records"), "written": f"outputs/{STEM}_dataset_manifest.json"})\n\n\n# --------------------------------------------------------------------------------------------------\n# model stages\n# --------------------------------------------------------------------------------------------------\n\n\ndef base_pipeline(run: Run, P):\n    return P.SAM2SegmentationPipeline.from_pretrained(weights_dir=snapshot_dir(run, P), allow_download=False)\n\n\ndef adapted_pipeline(run: Run, P):\n    adapt = run.read_state("adapt.json", "adapter reload")\n    return P.SAM2SegmentationPipeline.from_artifact(Path(adapt["artifact_dir"]), weights_dir=snapshot_dir(run, P)), adapt\n\n\ndef evaluate_conditions(pipe, held: list[dict[str, Any]]) -> dict[str, Any]:\n    return {name: pipe.evaluate_adaptation(condition(held, name)) for name in CONDITIONS}\n\n\ndef stage_baseline(run: Run) -> None:\n    P = package(run.root)\n    held = load_records(run, "held_out", "baseline")\n    pipe = base_pipeline(run, P)\n    results = evaluate_conditions(pipe, held)\n    for name, result in results.items():\n        print({"condition": name, "mean_mask_iou": round(result["mean_mask_iou"], 6), "per_record": [round(v, 4) for v in result["per_record_mask_iou"]], "box_baseline_mean_iou": rounded(result["box_baseline_mean_iou"])})\n    run.write_state("baseline.json", {"ids": [r["id"] for r in held], "results": results, "device": pipe.device})\n\n\ndef stage_adapt(run: Run) -> None:\n    import time\n\n    import numpy as np\n\n    P = package(run.root)\n    epochs = int(run.options.get("epochs", 2))\n    learning_rate = float(run.options.get("learning_rate", 2e-5))\n    seed = int(run.options.get("seed", SEED))\n    train = load_records(run, "train", "adapt")\n    held = load_records(run, "held_out", "adapt")\n    unseen = load_records(run, "unseen", "adapt")[0]\n    pipe = base_pipeline(run, P)  # always from the pinned base, so a re-run compares cleanly (SAM-S3)\n    started = time.perf_counter()\n    counts = pipe.freeze_for_adaptation()\n    history = pipe.finetune(train, held, epochs=epochs, learning_rate=learning_rate, seed=seed)\n    seconds = round(time.perf_counter() - started, 2)\n    print({"parameters": counts, "device": pipe.device, "seconds": seconds, "weight_delta_l2": pipe.adaptation_config["weight_delta_l2"]})\n    for entry in history:\n        print(rounded(entry))\n    # The adapting process\'s own prediction on the unseen record is the reference Section 9\'s fresh reload must reproduce.\n    reference = pipe.segment(unseen["image"], points=unseen["points"], point_labels=unseen["point_labels"], box=unseen["box"], multimask=False)\n    np.save(run.state / "unseen_reference_mask.npy", reference["masks"])\n    artifact = run.out / ARTIFACT_DIRNAME\n    if artifact.exists():\n        if not artifact.is_dir():\n            raise RuntimeError(f"notebook-owned artifact path is not a directory: {artifact}")\n        shutil.rmtree(artifact)\n    revision = source_record(run).get("revision") or ""\n    if len(revision) != 40:\n        raise RuntimeError("source.json does not record the 40-hex repository revision the adapter must name as its producer; regenerate the notebook from the repository")\n    artifact = pipe.save_artifact(artifact, producer_revision=revision)\n    run.write_state("adapt.json", {"settings": {"epochs": epochs, "learning_rate": learning_rate, "seed": seed}, "parameters": counts, "history": history, "seconds": seconds, "weight_delta_l2": pipe.adaptation_config["weight_delta_l2"], "adaptation_config": pipe.adaptation_config, "artifact_dir": str(artifact), "unseen_reference_scores": reference["iou_scores"], "device": pipe.device})\n    print({"adapter": f"outputs/{ARTIFACT_DIRNAME}", "note": "Sections 8-10 attach this adapter to a fresh pinned base in their own processes"})\n\n\ndef interpretation(base: float, adapted: float, improved: int, n: int, name: str) -> str:\n    if base >= SATURATED_IOU:\n        return f"{name}: no headroom — the pretrained model already scores {base:.4f} mean IoU, so the adapted {adapted:.4f} ({adapted - base:+.4f}, {improved}/{n} records improved) is a mechanics check, not evidence that adaptation helped"\n    return f"{name}: pretrained {base:.4f} → adapted {adapted:.4f} ({adapted - base:+.4f}); {improved} of {n} held-out records improved. One holdout of {n} records with no dispersion estimate: read the direction and the per-record pattern, not the size"\n\n\ndef stage_evaluate(run: Run) -> None:\n    P = package(run.root)\n    data = run.read_state("data.json", "evaluate")\n    baseline = run.read_state("baseline.json", "evaluate")\n    held = load_records(run, "held_out", "evaluate")\n    pipe, adapt = adapted_pipeline(run, P)\n    adapted = evaluate_conditions(pipe, held)\n    summary, readings = {}, []\n    for name in CONDITIONS:\n        before = baseline["results"][name]["per_record_mask_iou"]\n        after = adapted[name]["per_record_mask_iou"]\n        rows = [{"id": i, "pretrained": round(b, 6), "adapted": round(a, 6), "change": round(a - b, 6)} for i, b, a in zip(baseline["ids"], before, after, strict=True)]\n        improved = sum(a > b for b, a in zip(before, after, strict=True))\n        base_mean, adapted_mean = baseline["results"][name]["mean_mask_iou"], adapted[name]["mean_mask_iou"]\n        reading = interpretation(base_mean, adapted_mean, improved, len(rows), name)\n        readings.append(reading)\n        summary[name] = {"prompts": CONDITIONS[name], "pretrained_mean_iou": base_mean, "adapted_mean_iou": adapted_mean, "change": adapted_mean - base_mean, "records_improved": improved, "records": len(rows), "per_record": rows, "headroom_before": 1.0 - base_mean}\n        print({"condition": name, "pretrained_mean_iou": round(base_mean, 6), "adapted_mean_iou": round(adapted_mean, 6), "change": round(adapted_mean - base_mean, 6), "records_improved": f"{improved}/{len(rows)}"})\n        for row in rows:\n            print(row)\n    box_baseline = adapted["point+box"]["box_baseline_mean_iou"]\n    print({"prompt_box_fill_baseline_mean_iou": rounded(box_baseline), "note": "filling the prompt box shows what prompting adds; it is not the adaptation comparison"})\n    for reading in readings:\n        print({"interpretation": reading})\n    report = {\n        "task": "promptable-image-segmentation-adaptation",\n        "verdict": "sample-sanity",\n        "dataset_kind": data["kind"],\n        "estimation": data["split"],\n        "conditions": summary,\n        "baseline_pretrained": baseline["results"]["point+box"],\n        "adapted": adapted["point+box"],\n        "prompt_box_fill_baseline_mean_iou": box_baseline,\n        "interpretation": readings,\n        "weights_moved_vs_quality_changed": "weight_delta_l2 > 0 shows the parameters moved; only the held-out per-record change can show whether quality changed",\n        "weight_delta_l2": adapt["weight_delta_l2"],\n        "computed_by": "a fresh process that attached the exported adapter to a fresh pinned base",\n    }\n    run.write_output(f"{STEM}_evaluation_report.json", report)\n    run.write_state("evaluate.json", {name: {k: summary[name][k] for k in ("pretrained_mean_iou", "adapted_mean_iou", "records_improved", "records")} for name in summary})\n    print({"report": f"outputs/{STEM}_evaluation_report.json"})\n\n\ndef stage_unseen(run: Run) -> None:\n    import numpy as np\n\n    P = package(run.root)\n    data = run.read_state("data.json", "unseen")\n    unseen = load_records(run, "unseen", "unseen")[0]\n    pipe, adapt = adapted_pipeline(run, P)\n    result = pipe.segment(unseen["image"], points=unseen["points"], point_labels=unseen["point_labels"], box=unseen["box"], multimask=False)\n    iou = P.mask_iou(result["masks"][0], unseen["mask"])\n    print({"record": data["unseen_source"], "mask_iou_sample_sanity": round(iou, 6), "model_iou_score": round(result["iou_scores"][0], 6)})\n    reference_mask = np.load(run.state / "unseen_reference_mask.npy")\n    mask_exact = bool(np.array_equal(reference_mask, result["masks"]))\n    scores_close = bool(np.allclose(adapt["unseen_reference_scores"], result["iou_scores"], rtol=RELOAD_RTOL, atol=RELOAD_ATOL))\n    summary = {"verification": "PASSED" if mask_exact and scores_close else "FAILED", "process": "fresh (separate from the adapting stage)", "mask_exact": mask_exact, "scores_within_tolerance": scores_close, "score_rtol": RELOAD_RTOL, "score_atol": RELOAD_ATOL, "artifact_dir": adapt["artifact_dir"]}\n    run.write_state("unseen.json", {"id": unseen["id"], "source": data["unseen_source"], "mask_iou": iou, "model_iou_score": result["iou_scores"][0], "reload": summary})\n    if not (mask_exact and scores_close):\n        raise RuntimeError(f"The adapter attached to a fresh base does not reproduce the adapting process (mask exact: {mask_exact}, scores within rtol {RELOAD_RTOL}: {scores_close}). Do not ship this artifact.")\n    manifest = json.loads((Path(adapt["artifact_dir"]) / P.ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))\n    print({"artifact_format": manifest.get("format"), "files": sorted(p.name for p in Path(adapt["artifact_dir"]).iterdir())})\n    print(summary)\n\n\ndef stage_result(run: Run) -> None:\n    import importlib.metadata\n    import platform\n\n    P = package(run.root)\n    data = run.read_state("data.json", "result")\n    adapt = run.read_state("adapt.json", "result")\n    unseen = run.read_state("unseen.json", "result")\n    source = source_record(run)\n    payload = {\n        "dataset": json.loads((run.out / f"{STEM}_dataset_manifest.json").read_text(encoding="utf-8")),\n        "adaptation": adapt["adaptation_config"],\n        "evaluation": json.loads((run.out / f"{STEM}_evaluation_report.json").read_text(encoding="utf-8")),\n        "unseen": {"id": unseen["id"], "source": unseen["source"], "mask_iou": unseen["mask_iou"]},\n        "artifact": unseen["reload"],\n        "notebook_source": source,\n        "repository_revision": source.get("revision"),\n        "base_model": {"id": P.MODEL_ID, "revision": P.MODEL_REVISION, "license": P.MODEL_LICENSE},\n        "runtime": {"python": platform.python_version(), **{d: importlib.metadata.version(d) for d in ("torch", "transformers", "safetensors")}, "device": adapt["device"]},\n    }\n    run.write_output(f"{STEM}_result.json", payload)\n    print({"status": "E2E COMPLETE", "dataset_kind": data["kind"], "optimizer_steps": sum(item.get("optimizer_steps", 0) for item in adapt["history"]), "weight_delta_l2": round(adapt["weight_delta_l2"], 6), "reload": unseen["reload"]["verification"], "runtime": payload["runtime"]})\n    print("outputs/:")\n    for path in sorted(run.out.rglob("*")):\n        if path.is_file():\n            print(f"  - {path.relative_to(run.out.parent).as_posix()} ({path.stat().st_size / 1024:.1f} KB)")\n\n\ndef stage_activity(run: Run) -> None:\n    """SAM-M3: re-run the fine-tune from the pinned base with a different learning rate and print both prompt conditions\n    beside the pretrained and default results; writes only to outputs/activity/."""\n    P = package(run.root)\n    adapt = run.read_state("adapt.json", "activity")\n    baseline = run.read_state("baseline.json", "activity")\n    default = run.read_state("evaluate.json", "activity")\n    train = load_records(run, "train", "activity")\n    held = load_records(run, "held_out", "activity")\n    learning_rate = float(run.options.get("learning_rate", 2e-4))\n    before = sorted(p.name for p in run.out.glob(f"{STEM}_*"))\n    pipe = base_pipeline(run, P)\n    pipe.freeze_for_adaptation()\n    history = pipe.finetune(train, held, epochs=adapt["settings"]["epochs"], learning_rate=learning_rate, seed=adapt["settings"]["seed"])\n    results = evaluate_conditions(pipe, held)\n    rows = []\n    for name in CONDITIONS:\n        rows.append({"condition": name, "pretrained": round(baseline["results"][name]["mean_mask_iou"], 6), f"default (lr {adapt[\'settings\'][\'learning_rate\']:g})": round(default[name]["adapted_mean_iou"], 6), f"activity (lr {learning_rate:g})": round(results[name]["mean_mask_iou"], 6), "activity_records_improved": f"{sum(a > b for a, b in zip(results[name][\'per_record_mask_iou\'], baseline[\'results\'][name][\'per_record_mask_iou\'], strict=True))}/{len(held)}"})\n    for row in rows:\n        print(row)\n    print({"activity_history": rounded(history), "weight_delta_l2": round(pipe.adaptation_config["weight_delta_l2"], 6)})\n    run.write_output(f"activity/{STEM}_activity_lr_{learning_rate:g}.json", {"learning_rate": learning_rate, "rows": rows, "history": history, "weight_delta_l2": pipe.adaptation_config["weight_delta_l2"]})\n    if sorted(p.name for p in run.out.glob(f"{STEM}_*")) != before:\n        raise RuntimeError("the activity changed the canonical outputs; it must write only to outputs/activity/")\n    print(f"wrote outputs/activity/{STEM}_activity_lr_{learning_rate:g}.json (the canonical outputs and the adapter are unchanged)")\n\n\nSTAGES = {\n    "weights": stage_weights,\n    "data": stage_data,\n    "validate": stage_validate,\n    "baseline": stage_baseline,\n    "adapt": stage_adapt,\n    "evaluate": stage_evaluate,\n    "unseen": stage_unseen,\n    "result": stage_result,\n    "activity": stage_activity,\n}\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--outputs", type=Path, required=True)\n    parser.add_argument("--weights", type=Path, required=True)\n    parser.add_argument("--stage", choices=sorted(STAGES), required=True)\n    parser.add_argument("--options", default="{}")\n    args = parser.parse_args(argv)\n    run = Run(args.root.resolve(), args.outputs.resolve(), args.weights.resolve(), json.loads(args.options))\n    error_file = run.state / f"{args.stage}.error.json"\n    error_file.unlink(missing_ok=True)\n    try:\n        STAGES[args.stage](run)\n    except BaseException as exc:  # noqa: BLE001 -- every failure is reported to the kernel with its own message\n        traceback.print_exc()\n        error_file.write_text(json.dumps({"type": type(exc).__name__, "message": str(exc)}), encoding="utf-8")\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'requirements.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via requests\ncharset-normalizer==3.5.2 \\\n    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \\\n    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \\\n    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \\\n    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \\\n    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \\\n    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \\\n    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \\\n    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \\\n    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \\\n    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \\\n    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \\\n    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \\\n    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \\\n    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \\\n    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \\\n    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \\\n    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \\\n    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \\\n    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \\\n    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \\\n    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \\\n    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \\\n    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \\\n    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \\\n    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \\\n    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \\\n    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \\\n    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \\\n    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \\\n    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \\\n    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \\\n    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \\\n    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \\\n    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \\\n    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \\\n    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \\\n    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \\\n    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \\\n    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \\\n    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \\\n    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \\\n    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \\\n    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \\\n    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \\\n    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \\\n    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \\\n    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \\\n    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \\\n    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \\\n    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \\\n    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \\\n    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \\\n    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \\\n    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \\\n    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \\\n    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \\\n    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \\\n    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \\\n    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \\\n    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \\\n    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \\\n    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \\\n    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \\\n    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \\\n    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \\\n    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \\\n    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \\\n    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \\\n    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \\\n    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \\\n    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \\\n    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \\\n    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \\\n    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \\\n    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \\\n    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \\\n    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \\\n    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \\\n    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \\\n    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \\\n    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \\\n    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \\\n    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \\\n    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \\\n    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \\\n    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \\\n    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \\\n    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \\\n    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \\\n    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \\\n    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \\\n    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \\\n    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \\\n    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \\\n    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \\\n    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \\\n    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \\\n    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \\\n    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \\\n    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \\\n    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \\\n    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \\\n    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \\\n    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \\\n    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \\\n    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \\\n    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \\\n    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \\\n    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \\\n    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \\\n    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \\\n    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \\\n    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \\\n    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \\\n    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \\\n    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \\\n    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \\\n    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \\\n    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \\\n    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \\\n    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \\\n    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \\\n    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \\\n    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \\\n    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \\\n    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \\\n    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \\\n    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \\\n    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \\\n    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \\\n    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \\\n    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \\\n    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \\\n    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \\\n    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \\\n    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \\\n    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \\\n    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \\\n    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \\\n    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \\\n    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \\\n    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \\\n    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \\\n    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \\\n    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \\\n    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \\\n    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \\\n    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \\\n    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \\\n    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \\\n    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \\\n    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \\\n    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \\\n    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \\\n    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \\\n    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \\\n    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \\\n    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \\\n    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \\\n    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \\\n    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \\\n    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \\\n    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \\\n    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \\\n    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \\\n    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \\\n    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \\\n    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \\\n    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \\\n    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \\\n    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \\\n    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f\n    # via requests\ncuda-bindings==13.4.3 \\\n    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n    # via torch\ncuda-pathfinder==1.8.3 \\\n    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f\n    # via cuda-bindings\ncuda-toolkit==13.0.3.0 \\\n    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f\n    # via torch\nfilelock==4.0.12 \\\n    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \\\n    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfsspec==2026.9.0 \\\n    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n    # via\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhuggingface-hub==0.36.2 \\\n    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n    # via\n    #   sam2-segmentation-pipeline (pyproject.toml)\n    #   tokenizers\n    #   transformers\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via requests\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\nmarkupsafe==3.0.4 \\\n    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \\\n    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \\\n    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \\\n    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \\\n    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \\\n    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \\\n    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \\\n    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \\\n    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \\\n    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \\\n    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \\\n    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \\\n    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \\\n    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \\\n    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \\\n    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \\\n    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \\\n    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \\\n    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \\\n    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \\\n    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \\\n    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \\\n    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \\\n    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \\\n    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \\\n    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \\\n    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \\\n    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \\\n    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \\\n    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \\\n    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \\\n    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \\\n    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \\\n    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \\\n    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \\\n    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \\\n    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \\\n    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \\\n    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \\\n    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \\\n    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \\\n    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \\\n    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \\\n    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \\\n    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \\\n    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \\\n    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \\\n    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \\\n    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \\\n    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \\\n    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \\\n    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \\\n    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \\\n    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \\\n    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \\\n    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \\\n    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \\\n    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \\\n    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \\\n    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \\\n    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \\\n    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \\\n    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \\\n    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \\\n    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \\\n    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \\\n    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \\\n    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \\\n    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \\\n    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \\\n    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \\\n    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \\\n    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \\\n    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \\\n    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \\\n    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \\\n    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \\\n    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \\\n    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \\\n    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \\\n    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \\\n    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \\\n    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \\\n    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \\\n    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \\\n    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \\\n    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \\\n    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \\\n    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \\\n    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \\\n    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \\\n    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \\\n    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \\\n    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \\\n    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \\\n    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \\\n    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \\\n    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \\\n    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \\\n    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \\\n    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \\\n    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \\\n    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \\\n    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \\\n    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \\\n    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \\\n    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \\\n    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \\\n    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \\\n    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \\\n    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \\\n    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \\\n    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \\\n    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \\\n    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \\\n    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \\\n    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \\\n    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \\\n    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \\\n    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \\\n    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \\\n    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \\\n    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \\\n    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \\\n    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \\\n    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \\\n    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \\\n    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \\\n    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \\\n    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \\\n    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \\\n    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \\\n    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \\\n    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \\\n    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \\\n    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \\\n    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \\\n    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \\\n    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \\\n    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \\\n    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \\\n    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \\\n    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \\\n    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \\\n    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \\\n    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \\\n    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \\\n    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \\\n    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \\\n    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \\\n    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3\n    # via jinja2\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nnumpy==2.5.3 \\\n    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \\\n    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \\\n    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \\\n    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \\\n    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \\\n    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \\\n    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \\\n    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \\\n    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \\\n    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \\\n    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \\\n    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \\\n    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \\\n    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \\\n    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \\\n    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \\\n    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \\\n    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \\\n    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \\\n    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \\\n    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \\\n    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \\\n    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \\\n    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \\\n    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \\\n    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \\\n    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \\\n    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \\\n    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \\\n    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \\\n    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \\\n    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \\\n    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \\\n    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \\\n    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \\\n    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \\\n    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \\\n    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \\\n    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \\\n    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \\\n    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \\\n    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \\\n    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \\\n    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \\\n    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \\\n    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \\\n    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \\\n    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \\\n    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \\\n    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \\\n    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \\\n    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \\\n    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \\\n    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \\\n    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \\\n    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \\\n    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \\\n    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \\\n    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \\\n    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \\\n    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \\\n    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \\\n    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \\\n    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \\\n    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \\\n    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab\n    # via\n    #   sam2-segmentation-pipeline (pyproject.toml)\n    #   torchvision\n    #   transformers\nnvidia-cublas==13.1.1.3 \\\n    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \\\n    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \\\n    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5\n    # via\n    #   cuda-toolkit\n    #   nvidia-cudnn-cu13\n    #   nvidia-cusolver\nnvidia-cuda-cupti==13.0.85 \\\n    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n    # via cuda-toolkit\nnvidia-cuda-nvrtc==13.0.88 \\\n    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n    # via\n    #   cuda-toolkit\n    #   nvidia-cublas\nnvidia-cuda-runtime==13.0.96 \\\n    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n    # via cuda-toolkit\nnvidia-cudnn-cu13==9.24.0.43 \\\n    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \\\n    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \\\n    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f\n    # via torch\nnvidia-cufft==12.0.0.61 \\\n    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n    # via cuda-toolkit\nnvidia-cufile==1.15.1.6 \\\n    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n    # via cuda-toolkit\nnvidia-curand==10.4.0.35 \\\n    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n    # via cuda-toolkit\nnvidia-cusolver==12.0.4.66 \\\n    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n    # via cuda-toolkit\nnvidia-cusparse==12.6.3.3 \\\n    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n    # via\n    #   cuda-toolkit\n    #   nvidia-cusolver\nnvidia-cusparselt-cu13==0.8.1 \\\n    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \\\n    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \\\n    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215\n    # via torch\nnvidia-nccl-cu13==2.30.7 \\\n    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \\\n    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50\n    # via torch\nnvidia-nvjitlink==13.4.92 \\\n    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \\\n    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \\\n    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \\\n    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323\n    # via\n    #   cuda-toolkit\n    #   nvidia-cufft\n    #   nvidia-cusolver\n    #   nvidia-cusparse\nnvidia-nvshmem-cu13==3.4.5 \\\n    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n    # via torch\nnvidia-nvtx==13.0.85 \\\n    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n    # via cuda-toolkit\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   huggingface-hub\n    #   transformers\npillow==11.3.0 \\\n    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \\\n    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \\\n    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \\\n    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \\\n    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \\\n    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \\\n    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \\\n    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \\\n    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \\\n    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \\\n    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \\\n    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \\\n    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \\\n    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \\\n    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \\\n    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \\\n    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \\\n    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \\\n    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \\\n    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \\\n    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \\\n    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \\\n    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \\\n    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \\\n    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \\\n    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \\\n    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \\\n    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \\\n    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \\\n    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \\\n    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \\\n    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \\\n    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \\\n    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \\\n    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \\\n    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \\\n    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \\\n    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \\\n    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \\\n    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \\\n    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \\\n    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \\\n    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \\\n    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \\\n    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \\\n    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \\\n    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \\\n    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \\\n    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \\\n    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \\\n    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \\\n    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \\\n    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \\\n    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \\\n    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \\\n    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \\\n    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \\\n    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \\\n    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \\\n    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \\\n    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \\\n    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \\\n    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \\\n    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \\\n    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \\\n    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \\\n    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \\\n    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \\\n    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \\\n    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \\\n    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \\\n    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \\\n    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \\\n    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \\\n    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \\\n    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \\\n    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \\\n    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \\\n    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \\\n    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \\\n    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \\\n    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \\\n    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \\\n    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \\\n    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \\\n    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \\\n    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \\\n    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \\\n    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \\\n    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \\\n    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \\\n    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \\\n    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \\\n    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \\\n    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \\\n    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \\\n    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \\\n    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \\\n    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \\\n    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \\\n    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \\\n    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \\\n    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \\\n    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \\\n    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \\\n    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3\n    # via\n    #   sam2-segmentation-pipeline (pyproject.toml)\n    #   torchvision\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   huggingface-hub\n    #   transformers\nregex==2026.9.29 \\\n    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \\\n    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \\\n    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \\\n    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \\\n    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \\\n    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \\\n    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \\\n    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \\\n    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \\\n    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \\\n    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \\\n    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \\\n    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \\\n    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \\\n    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \\\n    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \\\n    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \\\n    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \\\n    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \\\n    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \\\n    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \\\n    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \\\n    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \\\n    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \\\n    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \\\n    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \\\n    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \\\n    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \\\n    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \\\n    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \\\n    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \\\n    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \\\n    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \\\n    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \\\n    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \\\n    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \\\n    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \\\n    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \\\n    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \\\n    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \\\n    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \\\n    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \\\n    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \\\n    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \\\n    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \\\n    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \\\n    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \\\n    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \\\n    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \\\n    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \\\n    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \\\n    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \\\n    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \\\n    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \\\n    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \\\n    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \\\n    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \\\n    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \\\n    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \\\n    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \\\n    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \\\n    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \\\n    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \\\n    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \\\n    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \\\n    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \\\n    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \\\n    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \\\n    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \\\n    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \\\n    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \\\n    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \\\n    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \\\n    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \\\n    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \\\n    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \\\n    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \\\n    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \\\n    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \\\n    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \\\n    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \\\n    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \\\n    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \\\n    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \\\n    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \\\n    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \\\n    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \\\n    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \\\n    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \\\n    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \\\n    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \\\n    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \\\n    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \\\n    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \\\n    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \\\n    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \\\n    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \\\n    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \\\n    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \\\n    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \\\n    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \\\n    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \\\n    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \\\n    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \\\n    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \\\n    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \\\n    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \\\n    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \\\n    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \\\n    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \\\n    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \\\n    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \\\n    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \\\n    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \\\n    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \\\n    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \\\n    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \\\n    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \\\n    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \\\n    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \\\n    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \\\n    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \\\n    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \\\n    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \\\n    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \\\n    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \\\n    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \\\n    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \\\n    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \\\n    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52\n    # via transformers\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via\n    #   huggingface-hub\n    #   transformers\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via\n    #   sam2-segmentation-pipeline (pyproject.toml)\n    #   transformers\nsetuptools==84.0.0 \\\n    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n    # via torch\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\ntokenizers==0.22.2 \\\n    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n    # via transformers\ntorch==2.14.0 \\\n    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \\\n    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \\\n    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \\\n    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \\\n    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \\\n    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \\\n    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \\\n    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \\\n    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \\\n    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \\\n    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \\\n    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \\\n    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \\\n    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \\\n    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \\\n    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \\\n    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \\\n    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \\\n    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \\\n    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \\\n    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \\\n    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \\\n    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \\\n    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963\n    # via\n    #   sam2-segmentation-pipeline (pyproject.toml)\n    #   torchvision\ntorchaudio==2.11.0 \\\n    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \\\n    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \\\n    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \\\n    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \\\n    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \\\n    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \\\n    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \\\n    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \\\n    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \\\n    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \\\n    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \\\n    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \\\n    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \\\n    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \\\n    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \\\n    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \\\n    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \\\n    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \\\n    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \\\n    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \\\n    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \\\n    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \\\n    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \\\n    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \\\n    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \\\n    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \\\n    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \\\n    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84\n    # via sam2-segmentation-pipeline (pyproject.toml)\ntorchvision==0.29.0 \\\n    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \\\n    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \\\n    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \\\n    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \\\n    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \\\n    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \\\n    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \\\n    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \\\n    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \\\n    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \\\n    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \\\n    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \\\n    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \\\n    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \\\n    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \\\n    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \\\n    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \\\n    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \\\n    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \\\n    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \\\n    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \\\n    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \\\n    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \\\n    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e\n    # via sam2-segmentation-pipeline (pyproject.toml)\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.6 \\\n    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n    # via sam2-segmentation-pipeline (pyproject.toml)\ntriton==3.8.0 \\\n    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \\\n    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \\\n    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \\\n    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \\\n    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \\\n    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \\\n    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \\\n    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \\\n    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \\\n    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \\\n    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \\\n    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3\n    # via torch\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   huggingface-hub\n    #   torch\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via requests\n', 'weights/sam2.1-hiera-small/dimer-base-manifest.json': '{\n  "format": "dimer_hf_snapshot",\n  "formatVersion": 1,\n  "modelKey": "sam2.1-hiera-small",\n  "modelId": "facebook/sam2.1-hiera-small",\n  "revision": "ee5bba1d82bb8749febdf90f45e84b687142ba03",\n  "files": [\n    {\n      "path": "README.md",\n      "bytes": 19974,\n      "sha256": "6ec2d54879e41ad876d8cded0d641e8bc6ab74d5e095a73afc3f8406372ee6e9"\n    },\n    {\n      "path": "config.json",\n      "bytes": 5698,\n      "sha256": "97ff9f65b76d107acda4247885f0a5555d0048850ae3c5f97183df289aaecde9"\n    },\n    {\n      "path": "model.safetensors",\n      "bytes": 184305280,\n      "sha256": "0a4067b11ce1e23d5229203f11c718a823060d15a4b23fa2372a7d4b77cbbc60"\n    },\n    {\n      "path": "preprocessor_config.json",\n      "bytes": 683,\n      "sha256": "6ebf229ee259368ce4a8d4f2fe893a72b053023710853e257253939e601f583d"\n    },\n    {\n      "path": "processor_config.json",\n      "bytes": 95,\n      "sha256": "f8a68e865cfad115c1c2763f3d93eca7b1c622da06da2a9273eb437fb2389b6d"\n    },\n    {\n      "path": "sam2.1_hiera_s.yaml",\n      "bytes": 3761,\n      "sha256": "632e5cd0104f5ab6cd4f9d2dfd80a8e7240e481ad7960a13cad2ae3504b88dbd"\n    },\n    {\n      "path": "video_preprocessor_config.json",\n      "bytes": 705,\n      "sha256": "9fccfe5f464ec38c2f236d0e6a68e95511c80c22132fc2fa4b9f7b65f24fad95"\n    }\n  ],\n  "totalBytes": 184336196\n}', 'LICENSE': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or Derivative\n          Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2026 Kurt Valcorza\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'source.json': '{\n  "repository": "kurtvalcorza/sam2-segmentation-pipeline",\n  "revision": "f60fa852d6b0f2782491bee4112fa126edf74b9b",\n  "generator": "build_notebook.py/3.0",\n  "notebook_spec": "2.2",\n  "files": {\n    "src/sam2_segmentation_pipeline/__init__.py": "52bed6dff5c5ba6efbd587f10a28a2e3b9bab133030edae117b5f822bf054624",\n    "src/sam2_segmentation_pipeline/pipeline.py": "8c2a3aac5444f2bebe2a56202fd42ae80e539e1a1921e5eb9f7b8c12c98d0a3b",\n    "tutorial_stages.py": "d0b153cbe20a4e9982e68ab3bccaf769d32bcbf9266b42f826cab72dbd62da13",\n    "requirements.txt": "4bf4b3491c86c884f8eeeb17361c41711b282adc51cc1a69c3a89dd4c2dc2a5f",\n    "weights/sam2.1-hiera-small/dimer-base-manifest.json": "ccbf7408250a591a6dbef9782ef2a262997bdab4ee97512516b54797769b9a11",\n    "LICENSE": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138"\n  },\n  "sources": {\n    "src/sam2_segmentation_pipeline/__init__.py": "src/sam2_segmentation_pipeline/__init__.py",\n    "src/sam2_segmentation_pipeline/pipeline.py": "src/sam2_segmentation_pipeline/pipeline.py",\n    "tutorial_stages.py": "tools/tutorial_stages.py",\n    "requirements.txt": "tutorials/requirements-colab.lock.txt",\n    "weights/sam2.1-hiera-small/dimer-base-manifest.json": "weights/sam2.1-hiera-small/dimer-base-manifest.json",\n    "LICENSE": "LICENSE"\n  }\n}\n'}
CARRIED_BINARY = {}
CARRIED_HASHES = {'src/sam2_segmentation_pipeline/__init__.py': '52bed6dff5c5ba6efbd587f10a28a2e3b9bab133030edae117b5f822bf054624', 'src/sam2_segmentation_pipeline/pipeline.py': '8c2a3aac5444f2bebe2a56202fd42ae80e539e1a1921e5eb9f7b8c12c98d0a3b', 'tutorial_stages.py': 'd0b153cbe20a4e9982e68ab3bccaf769d32bcbf9266b42f826cab72dbd62da13', 'requirements.txt': '4bf4b3491c86c884f8eeeb17361c41711b282adc51cc1a69c3a89dd4c2dc2a5f', 'weights/sam2.1-hiera-small/dimer-base-manifest.json': 'ccbf7408250a591a6dbef9782ef2a262997bdab4ee97512516b54797769b9a11', 'LICENSE': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'source.json': '2c7d731eb1d9ec272d0df8bc5864a7c72185b47d0ba8c89c6758858a466cb2dd'}
for name, payload in [*((n, t.encode('utf-8')) for n, t in CARRIED_FILES.items()), *((n, base64.b64decode(b)) for n, b in CARRIED_BINARY.items())]:
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    if hashlib.sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried file integrity failure: ' + name + '. Do not edit this cell; regenerate the notebook from the repository.')
NOTEBOOK_SOURCE = json.loads((ROOT / 'source.json').read_text(encoding='utf-8'))
print({'carried_files': len(CARRIED_HASHES), 'verified': True, 'repository': NOTEBOOK_SOURCE['repository'], 'revision': NOTEBOOK_SOURCE['revision'], 'generator': NOTEBOOK_SOURCE['generator']})

{'carried_files': 7, 'verified': True, 'repository': 'kurtvalcorza/sam2-segmentation-pipeline', 'revision': 'f60fa852d6b0f2782491bee4112fa126edf74b9b', 'generator': 'build_notebook.py/3.0'}


**Infrastructure: the isolated environment.** This cell installs **nothing into the notebook kernel**. It downloads one pinned file — the `uv` 0.12.15 installer wheel, refused unless its size and SHA-256 match — creates a separate virtual environment with its own CPython 3.12.12, and installs `requirements.txt` into it with `--require-hashes`: every package must be the locked version and match a locked digest. The hosted runtime's own packages (Colab preloads its own NumPy, pandas and PyTorch) are never replaced, which is why no restart is needed. The environment is keyed on the lock's SHA-256: a second **Run all** in the same runtime reuses it and prints `environment_reused: True`. The first build downloads PyTorch with its CUDA libraries (a few GB) and typically takes a few minutes. The cell also defines `run_stage` and `load_record`, the two helpers the learner cells use: each learner cell runs one stage of the carried runner in its own process, streams what it prints, and stops with the stage's own error message if it fails.

**Expected result:** one dictionary with the generating revision, the isolated environment's Python, the `torch`, `transformers`, `safetensors` versions, whether CUDA is visible, the number of locked packages, whether an existing environment was reused, and the setup time. A failed download or a hash mismatch stops the cell; never remove a pin or a hash to get past one.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated locked environment and define the stage runner
import io
import urllib.error
import urllib.request
import zipfile

LOCK_SHA256 = '4bf4b3491c86c884f8eeeb17361c41711b282adc51cc1a69c3a89dd4c2dc2a5f'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
ENV_ROOT.mkdir(parents=True, exist_ok=True)
UV = ENV_ROOT / 'uv-0.12.15'
UV_MARK = ENV_ROOT / 'uv-0.12.15.sha256'  # digest of the binary extracted from the verified wheel
if not (UV.is_file() and UV_MARK.is_file() and hashlib.sha256(UV.read_bytes()).hexdigest() == UV_MARK.read_text().strip()):
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2 ** (attempt + 1))
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('uv 0.12.15 wheel size/hash mismatch: refusing to run it')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
        UV.write_bytes(archive.read(member))
    UV.chmod(0o700)
    UV_MARK.write_text(hashlib.sha256(UV.read_bytes()).hexdigest())
# Stage processes inherit no kernel Python path or start-up file and no credentials (every download is public). The
# kernel may export an inline matplotlib backend that the isolated environment cannot import; stages write files.
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1', UV_CACHE_DIR=str(ENV_ROOT / 'cache'), MPLBACKEND='Agg', PYTHONNOUSERSITE='1')
for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'GOOGLE_APPLICATION_CREDENTIALS'):
    ENV.pop(name, None)
# One environment per lock digest: a second Run all (or a re-run of this cell) reuses a complete matching environment.
VENV = ENV_ROOT / ('venv-' + LOCK_SHA256[:16])
READY = VENV / '.dimer-ready'
PYTHON = VENV / 'bin' / 'python'
environment_reused = READY.is_file() and READY.read_text().strip() == LOCK_SHA256 and PYTHON.is_file()
if not environment_reused:
    shutil.rmtree(VENV, ignore_errors=True)
    subprocess.run([str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(VENV)], env=ENV, check=True)
    subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes', *['--only-binary', ':all:'], '--index-url', 'https://pypi.org/simple', '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
    READY.write_text(LOCK_SHA256)
probe = subprocess.run([str(PYTHON), '-c', "import importlib.metadata, json, platform, torch; print(json.dumps({'python': platform.python_version(), **{d: importlib.metadata.version(d) for d in ['torch', 'transformers', 'safetensors']}, 'cuda': torch.cuda.is_available()}))"], env=ENV, check=True, capture_output=True, text=True)
RUNTIME = json.loads(probe.stdout.strip().splitlines()[-1])
print({'notebook_source': NOTEBOOK_SOURCE['revision'], **RUNTIME, 'locked_packages': 47, 'environment': str(VENV), 'environment_reused': environment_reused, 'setup_seconds': round(time.perf_counter() - SESSION_START)})
if not RUNTIME['cuda']:
    print('The isolated environment sees no CUDA GPU: every stage runs on the CPU (slower; see Prerequisites and Troubleshooting).')


def run_stage(stage, **options):
    """Run one stage of the carried runner in its own process with the isolated interpreter; stream its output."""
    if not (ROOT / 'tutorial_stages.py').is_file() or not PYTHON.is_file():
        raise RuntimeError(f'The run directory {ROOT} has no carried files, or the isolated environment is gone: run the Infrastructure cells again in order (Sections 1, 2 and 3), or choose Runtime > Run all.')
    log = ROOT / 'logs' / (stage + '.log')
    log.parent.mkdir(exist_ok=True)
    error_file = ROOT / 'state' / (stage + '.error.json')
    error_file.unlink(missing_ok=True)
    command = [str(PYTHON), '-u', str(ROOT / 'tutorial_stages.py'), '--root', str(ROOT), '--outputs', str(OUTPUTS), '--weights', str(WEIGHTS), '--stage', stage, '--options', json.dumps(options)]
    print('Running stage', repr(stage), 'in the isolated environment; log:', log, flush=True)
    with log.open('w', encoding='utf-8') as output:
        process = subprocess.Popen(command, env=ENV, cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            output.write(line)
            output.flush()
            if line.strip() and len(line) < 4000:
                print(line.rstrip(), flush=True)
        process.wait()
    if process.returncode:
        detail = 'see the log above'
        if error_file.is_file():
            error = json.loads(error_file.read_text(encoding='utf-8'))
            detail = error['type'] + ': ' + error['message']
        raise RuntimeError(f'Stage {stage!r} failed (exit {process.returncode}): {detail}')


def load_record(name):
    """A JSON record a stage wrote to OUTPUTS."""
    return json.loads((OUTPUTS / name).read_text(encoding='utf-8'))

{'notebook_source': 'f60fa852d6b0f2782491bee4112fa126edf74b9b', 'python': '3.12.12', 'torch': '2.14.0', 'transformers': '4.57.6', 'safetensors': '0.8.0', 'cuda': True, 'locked_packages': 47, 'environment': '/tmp/dimer_uv/venv-4bf4b3491c86c884', 'environment_reused': False, 'setup_seconds': 67}


## 3. Pin, stage and verify the model · [Engineering]

> **Infrastructure.** The next code cell is collapsed. It downloads one file and checks it; you may run it without studying its implementation.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the carried package and the 7-file manifest (path, byte size, SHA-256) — and the `weights` stage first checks that they agree. `stage_missing_files(..., allow_download=True)` then fetches exactly the absent entries from the Hugging Face Hub **at revision `ee5bba1d82bb…`** (never `main`, ~184 MB), and `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch. There is no fallback to a different download and no remote model code is executed.

**What to notice:** the model id, revision, licence and file count; `fetched` lists the checkpoint on a first run and is empty on a rerun (staging only fetches a missing file); then `verified_files`. A mismatch stops the cell with an error naming the file — see **Troubleshooting**, and never edit a manifest to get past one.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned checkpoint
run_stage('weights')

Running stage 'weights' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/weights.log


{'model_id': 'facebook/sam2.1-hiera-small', 'revision': 'ee5bba1d82bb8749febdf90f45e84b687142ba03', 'license': 'apache-2.0', 'files': 7, 'total_bytes': 184336196}


{'weights_dir': '/content/weights/sam2.1-hiera-small', 'fetched': ['README.md', 'config.json', 'model.safetensors', 'preprocessor_config.json', 'processor_config.json', 'sam2.1_hiera_s.yaml', 'video_preprocessor_config.json']}


{'verified_files': 7, 'revision': 'ee5bba1d82bb8749febdf90f45e84b687142ba03', 'total_bytes': 184336196}


## 4. Generate or supply a paired segmentation dataset · [Evaluation practice]

The default 24 deterministic 256×256 scenes contain one rounded rectangle or ellipse plus a distractor rectangle on a flat background. Each record has an RGB image, a boolean mask, a foreground point and a bounding box. Records 0–17 train; 18–23 are held out — fixed before any model runs, so nothing the model does can influence the split.

**Bring your own data.** Set `USE_BYOD = True` and `BYOD_ZIP_PATH` (or, in Colab, leave it empty for the upload dialog; on Kaggle a single attached ZIP is found automatically). The split is a seeded shuffle of 25 % of the records; set `GROUP_SEPARATOR` so records whose stems share a prefix (`scene1_a`, `scene1_b` with `_`) stay together, otherwise every image is treated as independent. A wrapping folder, unmatched stems (named), JPEG masks, unsafe paths, oversized archives and fewer than 8 pairs each stop this cell with their own message.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ''  # @param {type:"string"}
GROUP_SEPARATOR = ''  # @param {type:"string"}
SEED = 42  # @param {type:"integer"}

def upload_one(what, field):
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError(f'{field} is empty, and the upload dialog exists only in Google Colab: set {field} to {what} in this runtime.') from None
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f'The upload was cancelled or empty: no file was received. Run this cell again and choose {what}, or set {field}.')
    if len(uploaded) != 1:
        raise ValueError(f'Upload exactly one file ({what}); got {sorted(uploaded)}.')
    upload_name, payload = next(iter(uploaded.items()))
    path = ROOT / 'inputs' / Path(upload_name).name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    return str(path)

byod_zip = ''
if USE_BYOD:
    byod_zip = BYOD_ZIP_PATH or ('' if Path('/kaggle/input').is_dir() else upload_one('one BYOD ZIP with images/ and masks/', 'BYOD_ZIP_PATH'))
run_stage('data', use_byod=USE_BYOD, byod_zip_path=byod_zip, group_separator=GROUP_SEPARATOR, seed=SEED)

Running stage 'data' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/data.log


{'dataset_kind': 'generated', 'origin': 'deterministic generated shape scenes (seeds 8100..8123)', 'records': 24, 'train': 18, 'held_out': 6}


{'split': 'fixed generated split: generated-00..17 train, generated-18..23 held out (the recorded split)'}


{'unseen': 'a separately generated scene (generated-24) outside the train and held-out index range'}


**What to notice:** `dataset_kind: 'generated'`, 24 records split 18 / 6, the split rule, and the unseen scene (`generated-24`) outside both.

## 5. Validate the dataset and split · [Evaluation practice]

Both splits must have unique ids, exact image/mask alignment, boolean masks with foreground and background, and valid prompts (the point inside the mask, the box overlapping it); each record gets a content fingerprint. Ids on both sides of the split stop the stage. The manifest is written to `outputs/sam2_segmentation_dataset_manifest.json`.

In [ ]:
run_stage('validate')

Running stage 'validate' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/validate.log


{


  "kind": "generated",


  "split": "fixed generated split: generated-00..17 train, generated-18..23 held out (the recorded split)",


  "overlap_ids": []


}


{'train_records': 18, 'held_out_records': 6, 'written': 'outputs/sam2_segmentation_dataset_manifest.json'}


**What to notice:** `overlap_ids: []`, 18 training and 6 held-out records, and the split rule repeated in the manifest.

## 6. Measure the pretrained held-out baseline · [Evaluation practice]

Before any update the pretrained model scores the six held-out records under two prompt conditions: **point+box** (the exact box and an interior point — the recorded condition) and **point-only** (the same point, no box). Each record's IoU is printed, with the IoU of simply filling the prompt box. Loading the model prints *You are using a model of type sam2_video to instantiate a model of type sam2*: the published `config.json` declares the video model, and image-mode `Sam2Model` loading the same weights is the intended use (upstream README), so the warning is expected.

**Predict before running:** under point+box prompts, how close to 1.0 will the pretrained model's mean IoU be on these scenes? And under point-only?

In [ ]:
run_stage('baseline')

Running stage 'baseline' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/baseline.log


You are using a model of type sam2_video to instantiate a model of type sam2. This is not supported for all configurations of models and can yield errors.


{'condition': 'point+box', 'mean_mask_iou': 0.998254, 'per_record': [1.0, 1.0, 1.0, 0.9974, 0.9924, 0.9997], 'box_baseline_mean_iou': 0.856681}


{'condition': 'point-only', 'mean_mask_iou': 0.998396, 'per_record': [0.9993, 1.0, 0.9991, 0.9995, 0.9924, 1.0], 'box_baseline_mean_iou': None}


**What to notice:** point+box mean IoU 0.998254 in the recorded run (every record near 1.0), the prompt-box fill baseline 0.856681, and the point-only row — also near 1.0.

**Checkpoint:** what is the most the fine-tune could gain under point+box prompts here?

<details>
<summary>Check your reasoning (open after answering)</summary>

At most 1 − 0.998254 ≈ 0.0017 mean IoU — the pretrained model is at ceiling, because one bright shape on a flat background with its exact box is easy for a promptable segmenter. Any change Section 8 shows under that condition is too small to read on six records. Point-only prompts remove the box, so the model must find the extent itself — but one interior point is enough on a single high-contrast shape, and that baseline sits near 0.998 too, so neither condition leaves room to show an effect here. The box-fill baseline (0.857) measures what prompting adds, not what adaptation adds.

</details>

## 7. Freeze the base and run bounded fine-tuning · [Concept]

The stage loads a fresh pinned base (so re-running this cell always starts from the base, never from an earlier adapter), freezes everything except the first mask-token output hypernetwork (139,808 parameters), and runs AdamW for `EPOCHS` epochs at `LEARNING_RATE` with batch size one on the 18 training records, with point+box prompts. Held-out IoU is computed after each epoch for **monitoring only**; the final weights are kept. A non-zero L2 weight delta is required — it shows the parameters moved, not that anything improved. The stage then exports the adapter to `outputs/sam2-mask-hypernetwork-adapter-v1/` and records its own prediction on the unseen scene, which Section 9's fresh reload must reproduce.

**Predict before running:** will the training loss fall? Will the per-epoch held-out IoU move much?

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 2e-5  # @param {type:"number"}
run_stage('adapt', epochs=EPOCHS, learning_rate=LEARNING_RATE, seed=SEED)

Running stage 'adapt' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/adapt.log


You are using a model of type sam2_video to instantiate a model of type sam2. This is not supported for all configurations of models and can yield errors.


{'parameters': {'trainable_parameters': 139808, 'frozen_parameters': 38398769}, 'device': 'cuda:0', 'seconds': 5.78, 'weight_delta_l2': 0.10881876898456387}


{'epoch': 1, 'train_loss': 0.007102, 'optimizer_steps': 18, 'val_mask_iou': 0.998383}


{'epoch': 2, 'train_loss': 0.005357, 'optimizer_steps': 18, 'val_mask_iou': 0.998457}


{'adapter': 'outputs/sam2-mask-hypernetwork-adapter-v1', 'note': 'Sections 8-10 attach this adapter to a fresh pinned base in their own processes'}


**What to notice:** 139,808 trainable parameters, 36 optimizer steps, training loss 0.0071 → 0.0054 and held-out IoU 0.998383 → 0.998457 across the two epochs in the recorded run, and a weight delta around 0.109.

<details>
<summary>Check your reasoning (open after running)</summary>

The loss falls — optimisation works — but the held-out IoU barely moves because it started at ceiling. A falling loss and a non-zero weight delta are optimisation evidence; held-out quality is measured in Section 8.

</details>

## 8. Evaluate the adapted model, record by record · [Evaluation practice]

A fresh process attaches the exported adapter to a fresh pinned base and scores the six held-out records under both prompt conditions. For each condition the stage prints the pretrained and adapted mean, the change, how many records improved, every record before and after, and an **interpretation** line: when the pretrained mean is 0.99 or above it says *no headroom* and calls the comparison a mechanics check. The report, with every per-record value, is written to `outputs/sam2_segmentation_evaluation_report.json`. These are sample-sanity results on six generated records: one holdout, no dispersion estimate.

**Predict before running:** under point+box, will the adapted mean differ from 0.998254 by more than 0.001? How many of the six records will improve?

In [ ]:
run_stage('evaluate')

Running stage 'evaluate' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/evaluate.log


You are using a model of type sam2_video to instantiate a model of type sam2. This is not supported for all configurations of models and can yield errors.


{'condition': 'point+box', 'pretrained_mean_iou': 0.998254, 'adapted_mean_iou': 0.998457, 'change': 0.000204, 'records_improved': '3/6'}


{'id': 'generated-18', 'pretrained': 1.0, 'adapted': 0.99986, 'change': -0.00014}


{'id': 'generated-19', 'pretrained': 1.0, 'adapted': 1.0, 'change': 0.0}


...

{'id': 'generated-23', 'pretrained': 1.0, 'adapted': 1.0, 'change': 0.0}


{'prompt_box_fill_baseline_mean_iou': 0.856681, 'note': 'filling the prompt box shows what prompting adds; it is not the adaptation comparison'}


{'interpretation': 'point+box: no headroom — the pretrained model already scores 0.9983 mean IoU, so the adapted 0.9985 (+0.0002, 3/6 records improved) is a mechanics check, not evidence that adaptation helped'}


{'interpretation': 'point-only: no headroom — the pretrained model already scores 0.9984 mean IoU, so the adapted 0.9981 (-0.0003, 0/6 records improved) is a mechanics check, not evidence that adaptation helped'}


{'report': 'outputs/sam2_segmentation_evaluation_report.json'}


**What to notice:** point+box 0.998254 → 0.998457 in the recorded run with the *no headroom* interpretation, the per-record rows, and the point-only rows, which carry the same *no headroom* interpretation.

**Checkpoint:** a classmate says "fine-tuning improved SAM from 0.99825 to 0.99846". What would you reply?

<details>
<summary>Check your reasoning (open after answering)</summary>

That the difference is +0.0002 mean IoU on six records that were already at ceiling: it is within what any small perturbation would produce, and the weights moving (Section 7) does not show quality changed. The run shows the adaptation machinery works. To claim improvement you would need data with headroom — textured backgrounds, low contrast, occlusion or loose boxes; point-only prompts do not supply it on these scenes — a change that holds across most records, and more than one holdout.

</details>

## 9. Infer on an unseen scene and verify a fresh reload · [Engineering]

A fresh process attaches the adapter to a fresh pinned base and segments the unseen scene — `generated-24`, outside the train and held-out range, or one of your held-out records with BYOD. It then compares the mask and scores with the prediction the adapting process recorded in Section 7: the mask must match exactly and the scores within `rtol=1e-5`, `atol=1e-6`, or the stage stops with *Do not ship this artifact*. The stage only reads the adapter, so re-running it alone is safe.

In [ ]:
run_stage('unseen')

Running stage 'unseen' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/unseen.log


You are using a model of type sam2_video to instantiate a model of type sam2. This is not supported for all configurations of models and can yield errors.


{'record': 'a separately generated scene (generated-24) outside the train and held-out index range', 'mask_iou_sample_sanity': 1.0, 'model_iou_score': 0.992438}


{'artifact_format': 'org.valcorza.sam2.mask-decoder-adapter', 'files': ['adapter.safetensors', 'manifest.json']}


{'verification': 'PASSED', 'process': 'fresh (separate from the adapting stage)', 'mask_exact': True, 'scores_within_tolerance': True, 'score_rtol': 1e-05, 'score_atol': 1e-06, 'artifact_dir': '/content/outputs/sam2-mask-hypernetwork-adapter-v1'}


**What to notice:** the unseen-scene IoU (1.0 in the recorded run, model score 0.992), the adapter's files, and `verification: 'PASSED'` with `mask_exact: True`.

**Checkpoint:** why compare against the adapting process instead of against the ground-truth mask?

<details>
<summary>Check your reasoning (open after answering)</summary>

The ground truth tests quality; this check tests packaging. A user receives the adapter and the pinned base, not the trained object, so the question is whether those files rebuild exactly the model that was evaluated. An exact mask and matching scores answer it; a mismatch would mean the evaluation describes a different model from the one shipped.

</details>

## 10. Export provenance and terminal summary · [Engineering]

`outputs/sam2_segmentation_result.json` binds the dataset manifest, the adaptation configuration and lineage, the evaluation report, the unseen-scene result, the reload evidence, the notebook source, the base model identity and licence, and the isolated environment's versions and device — without retaining any image.

In [ ]:
run_stage('result')

Running stage 'result' in the isolated environment; log: /content/dimer_runs/sam2_segmentation/8da0291f3dad/logs/result.log


{'status': 'E2E COMPLETE', 'dataset_kind': 'generated', 'optimizer_steps': 36, 'weight_delta_l2': 0.108819, 'reload': 'PASSED', 'runtime': {'python': '3.12.12', 'torch': '2.14.0', 'transformers': '4.57.6', 'safetensors': '0.8.0', 'device': 'cuda:0'}}


outputs/:


  - outputs/sam2-mask-hypernetwork-adapter-v1/adapter.safetensors (546.8 KB)


  - outputs/sam2-mask-hypernetwork-adapter-v1/manifest.json (3.2 KB)


  - outputs/sam2_segmentation_dataset_manifest.json (0.6 KB)


  - outputs/sam2_segmentation_evaluation_report.json (4.3 KB)


  - outputs/sam2_segmentation_result.json (9.7 KB)


## 11. Optional activity: does a larger learning rate help? · [Concept]

**Predict → Change → Run → Observe → Explain.** **Predict:** with `ACTIVITY_LEARNING_RATE = 2e-4` (10× the default), will the held-out IoU rise, fall or stay the same under each prompt condition? **Change:** tick `RUN_ACTIVITY` and set the rate. **Run** this cell: it fine-tunes a fresh copy of the base. **Observe** the table — pretrained, default, activity — for both conditions and the records improved. **Explain** what you see in terms of headroom. The activity writes only to `outputs/activity/`; the canonical outputs and the adapter are untouched.

In [ ]:
RUN_ACTIVITY = False  # @param {type:"boolean"}
ACTIVITY_LEARNING_RATE = 2e-4  # @param {type:"number"}
if RUN_ACTIVITY:
    run_stage('activity', learning_rate=ACTIVITY_LEARNING_RATE)
else:
    print('Optional activity skipped: tick RUN_ACTIVITY to run it. The canonical outputs are complete.')

Optional activity skipped: tick RUN_ACTIVITY to run it. The canonical outputs are complete.


**What to notice (if you ran it):** whether the larger rate moves either condition, and in which direction.

<details>
<summary>Check your reasoning (open after running)</summary>

No hosted run of the activity is recorded yet, so compare your rows. Under point+box there is almost no room to gain, and point-only sits at ceiling too, so a larger rate can mostly only hurt either condition; read the direction per record, not the size. A rate that improves the training loss but lowers held-out IoU is overfitting 18 easy scenes.

</details>

## Troubleshooting · [Engineering]

| Symptom | Likely cause | What to do |
|---|---|---|
| `No CUDA GPU detected, and this notebook's fine-tuning needs one` | a CPU-only runtime | *Runtime → Change runtime type → T4 GPU* (Colab) or turn on a GPU accelerator (Kaggle), then *Run all*. |
| `This notebook needs a Linux x86_64 runtime` | macOS, Windows or ARM kernel | Use Colab, Kaggle or a Linux x86_64 Jupyter kernel. |
| `Not enough free disk` | a used runtime | Start a fresh runtime. |
| `uv … wheel size/hash mismatch` or a `--require-hashes` error | a corrupted or substituted download | Re-run Section 2; never remove a pin or a hash. |
| a size or SHA-256 mismatch in Section 3 | a file that is not the pinned one | Delete `weights/sam2.1-hiera-small/` and re-run Section 3; never edit the manifest. |
| `You are using a model of type sam2_video to instantiate a model of type sam2` | the published config names the video model | Expected: image-mode loading of the same weights is the intended use. |
| `Stage '…' failed …: … is missing: run the stage that writes it` | a cell run out of order | Run the notebook in order from Section 4 (or *Run all*). |
| `BYOD_ZIP_PATH is empty, and the upload dialog exists only in Google Colab` | BYOD outside Colab with no path | Set `BYOD_ZIP_PATH`. |
| `/kaggle/input holds N ZIP files, not exactly one` | several attached ZIPs | Set `BYOD_ZIP_PATH` to the one you mean. |
| `The upload was cancelled or empty` / `Upload exactly one file` | a cancelled or multi-file upload | Run the cell again and choose one ZIP. |
| `<zip>: expected images/ and masks/ at the top level or inside one wrapping folder` | another layout | Put `images/` and `masks/` at the top level, or in one folder. |
| `<zip>: masks must be PNG` | JPEG masks (lossy, so edges blur) | Save masks as PNG, white = foreground. |
| `<zip>: every image needs a mask with the same stem; images without a mask: […]` | unmatched names | Rename so each `images/<stem>` has `masks/<stem>.png`. |
| `<zip>: unsafe member paths` / `… exceeds the 256 MiB BYOD ceiling` / `… 130-member` | an unsafe or oversized archive | Rebuild the ZIP with relative paths and at most 64 pairs. |
| `<zip>: found N image/mask pair(s); the E2E path needs at least 8` | too few pairs | Add pairs. |
| `the groups cannot be split into at least 2 training and 2 held-out records` | `GROUP_SEPARATOR` merges everything into one or two groups | Use a separator that matches your scene prefixes, or add scenes. |
| `The adapter attached to a fresh base does not reproduce the adapting process` | a broken export or a different base | Do not ship the adapter; re-run from Section 7. |

## Interpretation and limits

This run proves the exact notebook can validate paired masks, update the declared adapter surface from the pinned base, score a fixed held-out split per record under two prompt conditions, serialize only the adapter, attach it to the exact pinned base in a fresh process, and reproduce inference after reload. On the default point+box condition the pretrained model is already at ceiling, so that comparison is a mechanics check: it cannot show that adaptation helps, and the notebook says so. The point-only condition is at ceiling on these scenes as well, so it cannot show it either. Nothing here shows improvement on photographs or any deployment domain. Real use requires rights-cleared representative images, human-reviewed masks, leakage-safe (grouped) splits, and boundary-error analysis.

Successful execution proves that the recorded repository revision can complete this bounded tutorial without the repository being reachable at runtime. It does **not** establish benchmark superiority or production fitness.

## Conclusion · [Evaluation practice]

Write three sentences: how much headroom the pretrained model left under each prompt condition; what the per-record comparison does and does not show about the fine-tune; and what you would need before claiming that adaptation improves segmentation on your own images.

<details>
<summary>Sample conclusion (open after writing yours)</summary>

Under point+box prompts the pretrained model scored 0.998254 mean IoU on six held-out scenes, leaving almost no headroom, so the adapted 0.998457 (+0.0002) is a mechanics check: the hypernetwork trained, the adapter exported, and a fresh base plus the adapter reproduced the masks exactly. Point-only prompts were near ceiling too, so neither condition shows whether adaptation helps. Before claiming improvement on my own images I would need representative, human-reviewed masks, a grouped split, several seeds or folds for a dispersion estimate, and a condition where the pretrained model is clearly imperfect.

</details>

**Next experiments:** run the activity at 2e-4 and at 2e-6; bring your own ZIP with `GROUP_SEPARATOR` set and compare the two prompt conditions; change `EPOCHS` and watch the per-epoch held-out IoU.

## References

- Repository: https://github.com/kurtvalcorza/sam2-segmentation-pipeline
- Repository model card: https://github.com/kurtvalcorza/sam2-segmentation-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/facebook/sam2.1-hiera-small
- SAM 2 paper: https://arxiv.org/abs/2408.00714